# FlyLM-2 · P100 · SentencePiece와 MaleCNS 혼합 신경 동역학

`flylm-1-training.ipynb`의 전체 학습 흐름을 보존한 혼합 동역학 실험이다.
모든 `Traced` 뉴런과 그 사이의 원본 연결을 유지한다. 실제 sensory/motor annotation을 입출력 포트로 사용한다.
SentencePiece 토큰 → 작은 감각 변환기 → MaleCNS 회로 → 작은 운동 출력기 → 다음 토큰.
토크나이저를 train 데이터로 학습한 SentencePiece Unigram + byte fallback으로 변경한다.
대화 분할과 응답 전용 손실을 유지하며, 길이/학습량의 단위는 이제 바이트가 아닌 SentencePiece 토큰이다.
바이트 모델 체크포인트는 호환되지 않으므로 새로 학습해야 한다.

**이번 변경:** 문헌이 뒷받침하는 정확한 세포 유형에만 스파이크/연속 방식을 배정한다.
미확인 유형에는 기존 연속 방식을 명시적 기본값으로 사용한다. 근거집은 작은 시작 집합이며, 전체 문헌을 망라하지 않는다.
각 뉴런은 막 상태 `v`, 비음수 시냅스 신호 `s`, 느린 적응 `a`, 남은 불응기 `q`의 네 값을 유지한다.
스파이크 유형에는 이산 발화·막전위 리셋·불응기가 있고, 역전파에는 대리 미분을 사용한다.
시간 상수·역치·리셋·불응기의 수치는 공학적 설정이며 측정된 생리학 수치가 아니다. tick은 ms가 아니다.
질문 재주입·외부 언어모델·새로운 연결·전기 시냅스·수용체별 전도도·도파민 가소성은 추가하지 않는다.

**위에서 아래로 실행한다. v2 가중치는 v1 추론 모듈과 직접 호환되지 않는다.**
학습은 새로 시작하며 optimizer/데이터 위치의 정확한 재개를 제공하지 않는다.
기본 배치 2, TBPTT 64, SentencePiece 토큰당 4 tick, 최대 1,000 update는 실행 확인용이다. 언어 품질 향상을 보장하지 않는다.
`neuron_mode='graded_control'`로 바꾸면 모든 뉴런에 기존 연속식을 사용하는 대조군을 실행할 수 있다.
실험을 비교할 때 seed·데이터·배치·학습 노출량을 맞추고, 처리 속도와 신경 활동 진단도 함께 확인한다.

**Kaggle P100 실행:** GPU 가속기와 인터넷을 켜고 첫 코드 셀부터 실행한다.
P100에서 `sm_60`이 빠진 경우에만 공식 PyTorch CUDA 12.6 빌드로 교체한다.
이미 로드된 커널에 교체 전 버전이 남아 있을 때만 커널을 다시 시작한다.
FP32 CUDA CSR → 작은 그래프 검증 → 실제 전체 그래프의 64토큰 역전파/Adam 메모리 검사 → 학습 순서이다.
전체 그래프 메모리 검사는 가중치를 버리는 별도 시험이며, 학습 시작점에 영향을 주지 않는다.

입력: `/kaggle/input/datasets/chika1472/dailytalk` 아래 JSON.
출력: `/kaggle/working/flylm2_runs/<실행시각>`.
[MaleCNS 공식 데이터](https://male-cns.janelia.org/download/) ·
[공식 PyTorch 설치](https://pytorch.org/get-started/previous-versions/)

## 1. P100 설정과 혼합 동역학 실험 설정

전체 그래프 / **batch 2** / TBPTT 64 / SentencePiece 토큰당 4 tick / FP32.
v1의 뒤쪽 셀에서 사용하던 실제 배치 2를 여기에서 명시한다.
`vram_fraction=0.90`은 다른 GPU 사용량을 고려한 메모리 예산이다.
배치를 고정하면 한 번 검사한다. `batch_size='auto'`는 배치 1부터 두 배씩 늘려 검사하고,
예산에 맞는 마지막 검사 배치를 사용한다. 최적 처리 속도를 보장하는 탐색은 아니다.

`neuron_mode='mixed'`가 기본이며 `graded_control`은 기존 연속식 대조군이다.
미확인 유형의 `unknown_activity='graded'`는 과학적 판정이 아니라 기본 모델 선택이다.
스파이크 역치 중심 0.25, 학습 반경 0.20, 리셋 0, 불응기 2 tick, 대리 미분 기울기 5는 모두 공학적 가정이다.

**토크나이저:** `TOK_CFG`에서 어휘 수를 설정한다. 기본 요청값은 4096, 희귀 문자에는 256개 바이트 fallback을 사용한다.
최대 100,000개 train 쌍을 seed/hash로 선택해 어휘를 학습한다. `None`이면 모든 train 쌍을 사용한다.
전처리/어휘/특수 토큰/모델 해시를 캐시에 반영하고, 실제 어휘 수로 임베딩과 출력층을 만든다.
토큰당 4 tick을 유지하므로 문자당 계산량과 64토큰 기울기 구간의 텍스트 길이가 이전 버전과 달라진다.

In [ ]:
# 현재 커널에서 torch를 가져오기 전에 Pascal 지원을 확인한다.
import json
import subprocess
import sys

def disk_cuda_info():
    # 별도 프로세스는 커널에 캐시된 모듈 대신 설치된 파일을 확인한다.
    script = """import json, torch
print(json.dumps(dict(version=str(torch.__version__),
    capability=list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
    architectures=torch.cuda.get_arch_list())))
"""
    result = subprocess.run([sys.executable, '-c', script], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('설치된 PyTorch 확인 실패:\n' + result.stderr)
    return json.loads(result.stdout.strip().splitlines()[-1])

disk_info = disk_cuda_info()
print('Installed PyTorch:', disk_info)
if disk_info['capability'] == [6, 0] and 'sm_60' not in disk_info['architectures']:
    print('P100 지원이 없는 빌드입니다. 공식 CUDA 12.6 빌드로 교체합니다.')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch==2.10.0+cu126', 'torchvision==0.25.0+cu126', 'torchaudio==2.10.0+cu126',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    disk_info = disk_cuda_info()
    print('After installation:', disk_info)
    if 'sm_60' not in disk_info['architectures']:
        raise RuntimeError('설치 후에도 sm_60 지원이 없습니다. 위 pip 출력과 설치 정보를 확인하세요.')

loaded_torch = sys.modules.get('torch')
if (disk_info['capability'] == [6, 0] and loaded_torch is not None
        and 'sm_60' not in loaded_torch.cuda.get_arch_list()):
    raise RuntimeError(
        '디스크의 P100 지원 빌드는 확인됐지만 현재 커널에는 이전 torch가 로드돼 있습니다. '
        '커널을 한 번 재시작한 뒤 첫 셀부터 실행하세요. 다음 실행에서는 재설치하지 않습니다.'
    )
# Pascal 지원 확인 끝.

import importlib.util
import subprocess
import sys

required = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy',
            'pyarrow': 'pyarrow', 'tqdm': 'tqdm', 'sentencepiece': 'sentencepiece==0.2.2'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('CUDA PyTorch가 제공되는 Kaggle GPU 런타임을 선택하세요.')

import gc
import io
import sentencepiece as spm
import hashlib
import json
import math
import platform
import random
import shutil
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import scipy
from scipy import sparse
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

CFG = dict(seed=42, n_neurons=None, embedding_dim=32, brain_steps=4,
           batch_size=2, vram_fraction=0.90, max_batch_size=None, tbptt=64, prompt_grad_tokens=32,
           train_updates=1000, epochs=1, lr=0.001,
           neuron_mode='mixed', unknown_activity='graded',
           spike_threshold_center=0.25, spike_threshold_radius=0.20,
           spike_reset=0.0, refractory_ticks=2, surrogate_slope=5.0,
           checkpoint_tokens=True, nt_confidence=0.8, use_nt_signs=True,
           edge_block=262144,
           tau_initial=(2.0, 2.0, 64.0),
           tau_min=(0.5, 0.5, 8.0), tau_max=(32.0, 16.0, 512.0),
           log_every=50, eval_every=200, save_every=200,
           eval_max_pairs=4, eval_response_tokens=128)
# 작은 코퍼스에서는 실제 어휘 수가 요청값보다 작을 수 있다.
TOK_CFG = dict(vocab_size=4096, character_coverage=0.9995,
               max_training_pairs=100000, max_sentence_length=16384)
CACHE = Path('/kaggle/working/flylm_malecns_cache')
CACHE.mkdir(parents=True, exist_ok=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])

# 버전 문자열 대신 실제 CUDA 연산으로 런타임을 확인한다.
print(dict(torch=str(torch.__version__), cuda_build=torch.version.cuda,
           torch_file=torch.__file__, python=sys.executable))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 보이지 않습니다. Kaggle GPU 가속기 설정과 위 PyTorch 정보를 확인하세요.')
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print(dict(gpu=gpu_name, capability=capability, compiled_architectures=torch.cuda.get_arch_list()))
if 'P100' not in gpu_name:
    print('참고: 현재 GPU는 P100이 아닙니다:', gpu_name)
try:
    probe_row = torch.tensor([0, 1, 2], dtype=torch.int64, device=device)
    probe_col = torch.tensor([1, 0], dtype=torch.int64, device=device)
    probe_values = torch.tensor([2., 3.], device=device)
    probe_matrix = torch.sparse_csr_tensor(probe_row, probe_col, probe_values, size=(2, 2))
    probe_result = torch.sparse.mm(probe_matrix, torch.ones(2, 1, device=device))
    torch.testing.assert_close(probe_result, torch.tensor([[2.], [3.]], device=device))
    torch.cuda.synchronize()
except RuntimeError as error:
    raise RuntimeError(
        f'실제 CUDA CSR 실행이 실패했습니다: {error}\n'
        '위 torch/CUDA/GPU 정보와 원본 오류를 확인하세요. P100 지원 빌드 확인 이후의 실제 연산 오류입니다.'
    ) from error
print('PASS: FP32 CUDA CSR on', gpu_name, 'capability', capability)
del probe_row, probe_col, probe_values, probe_matrix, probe_result
# CUDA 런타임 검사 끝.

torch.manual_seed(CFG['seed'])
torch.cuda.manual_seed_all(CFG['seed'])

VERSIONS = dict(python=platform.python_version(), torch=torch.__version__,
                numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__, pyarrow=pa.__version__, sentencepiece=spm.__version__)
print(VERSIONS)
print(torch.cuda.get_device_name(0), 'GPU GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
assert CFG['tbptt'] == 64
assert CFG['batch_size'] == 'auto' or (isinstance(CFG['batch_size'], int) and CFG['batch_size'] >= 1)
assert 0 < CFG['vram_fraction'] < 1
assert CFG['max_batch_size'] is None or CFG['max_batch_size'] >= 1

assert CFG['epochs'] >= 1 and CFG['tbptt'] >= 1
assert CFG['train_updates'] is None or CFG['train_updates'] >= 1
assert min(CFG[k] for k in ['log_every', 'eval_every', 'save_every', 'eval_max_pairs', 'eval_response_tokens']) >= 1
assert 1 <= CFG['prompt_grad_tokens'] < CFG['tbptt']

## 1A. Kaggle JSON 디렉터리

`/kaggle/input/datasets/chika1472/dailytalk` 아래 JSON을 직접 읽는다.
ZIP 복사/압축 해제와 Google Drive 마운트는 없다.
파일 상대 경로·크기·SHA-256으로 캐시를 구분한다.
결과와 캐시는 쓰기 가능한 `/kaggle/working` 아래에 저장한다.
Kaggle 세션 종료 전 실행 결과를 노트북 출력으로 보존하거나 다운로드한다.

In [ ]:
DATA_CFG = dict(json_dir='/kaggle/input/datasets/chika1472/dailytalk',
                text_field='form', validation_fraction=0.05)
DATA_ROOT = Path(DATA_CFG['json_dir'])
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Kaggle에 데이터셋을 연결하세요: {DATA_ROOT}')
JSON_FILES = sorted(p for p in DATA_ROOT.rglob('*')
                    if p.is_file() and p.suffix.lower() == '.json'
                    and '__MACOSX' not in p.parts and not p.name.startswith('._'))
if not JSON_FILES:
    raise ValueError(f'JSON 파일이 없습니다: {DATA_ROOT}')
DATA_CACHE = Path('/kaggle/working/flylm_dialogue_cache')
DATA_CACHE.mkdir(parents=True, exist_ok=True)
dataset_sources = []
for path in tqdm(JSON_FILES, desc='JSON fingerprint'):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b''):
            digest.update(block)
    dataset_sources.append(dict(path=path.relative_to(DATA_ROOT).as_posix(),
                                bytes=path.stat().st_size, sha256=digest.hexdigest()))
dataset_sha256 = hashlib.sha256(json.dumps(dataset_sources, sort_keys=True).encode()).hexdigest()
print(dict(json_directory=str(DATA_ROOT), json_files=len(JSON_FILES), dataset_sha256=dataset_sha256))

## 1B. 대화 단위 분할 → train 전용 SentencePiece 학습 → 토큰 캐시

JSON 배열 순서와 두 화자의 인접 입력–답변 쌍, 같은 화자 연속 발화 합치기, 중복 제거 규칙을 유지한다.
`form`의 연속 공백만 정리하고 메타데이터는 본문에 넣지 않는다. 같은 대화가 train/validation에 걸치지 않도록 먼저 나눈다.
원문 쌍을 JSONL로 디스크에 저장한 뒤 **train 쌍만** SentencePiece Unigram 어휘 학습에 사용한다.
검증 문서가 토크나이저 학습에 섞이지 않는다. 큰 원문 전체를 메모리에 쌓지 않는다.

정규화는 `identity`, 자동 앞 공백은 끄고 공백을 보존한다. 실제 `▁` 문자는 바이트로 따로 인코딩한다.
byte fallback으로 희귀 한글·이모지·다른 언어를 표현한다. BOS=1, EOS=2, PAD=3, UNK=0이다.
코퍼스가 작으면 실제 어휘가 4096보다 작을 수 있다. 실제 크기를 체크포인트에 저장한다.

    BOS + SentencePiece("화자1: 입력 내용\n화자2: ") + SentencePiece(답변) + EOS
    └──────────────── loss 제외 ─────────────────┘  └── loss 적용 ──┘

prompt와 답변을 **각각 인코딩**해 경계를 넘는 subword 병합을 막고, 추론 prompt와 정확히 같은 토큰을 사용한다.
uint16 캐시의 길이·offset·prompt 경계를 새로 계산한다. padding에는 PAD를 사용하고 정답에서는 -100으로 가린다.
문자/어휘 캐시와 기존 바이트 체크포인트는 재사용하지 않는다.
[SentencePiece 공식 문서](https://github.com/google/sentencepiece) ·
[특수 토큰 설명](https://github.com/google/sentencepiece/blob/master/doc/special_symbols.md)

In [ ]:
def encode_text(tokenizer, text):
    # SentencePiece의 공백 표식과 실제 U+2581 문자를 구분해 원문을 보존한다.
    if '▁' not in text:
        return tokenizer.encode(text, out_type=int)
    parts = text.split('▁')
    ids = []
    literal = [tokenizer.piece_to_id(f'<0x{value:02X}>') for value in '▁'.encode('utf-8')]
    for i, part in enumerate(parts):
        if i:
            ids.extend(literal)
        ids.extend(tokenizer.encode(part, out_type=int))
    return ids


def render_prompt(question):
    return '화자1: ' + question + '\n화자2: '


def tokenizer_metadata(tokenizer):
    return dict(vocab_size=tokenizer.get_piece_size(), bos_id=tokenizer.bos_id(),
                eos_id=tokenizer.eos_id(), pad_id=tokenizer.pad_id(), unk_id=tokenizer.unk_id())


def validate_tokenizer(tokenizer, expected=None):
    meta = tokenizer_metadata(tokenizer)
    if (meta['bos_id'], meta['eos_id'], meta['pad_id'], meta['unk_id']) != (1, 2, 3, 0):
        raise ValueError('특수 토큰 설정이 다르다: BOS=1, EOS=2, PAD=3, UNK=0이어야 한다.')
    if not 260 <= meta['vocab_size'] <= 65536:
        raise ValueError('바이트 fallback과 uint16 캐시에 맞는 어휘 크기가 필요하다.')
    for value in range(256):
        piece = f'<0x{value:02X}>'
        index = tokenizer.piece_to_id(piece)
        if not tokenizer.is_byte(index) or tokenizer.id_to_piece(index) != piece:
            raise ValueError('모든 256개 바이트 fallback 토큰이 필요하다.')
    if expected is not None and any(expected[key] != value for key, value in meta.items()):
        raise ValueError('토크나이저의 어휘/특수 토큰 정보와 체크포인트 설정이 다르다.')
    return meta

def dialogue_turns(document, location):
    utterances = document.get('utterance')
    if not isinstance(utterances, list):
        raise ValueError(f'{location}: utterance must be a list')
    speakers, turns = {}, []
    for index, utterance in enumerate(utterances):
        if not isinstance(utterance, dict):
            raise ValueError(f'{location}: utterance[{index}] must be an object')
        text = utterance.get(DATA_CFG['text_field'])
        if not isinstance(text, str):
            raise ValueError(f'{location}: utterance[{index}].{DATA_CFG["text_field"]} must be a string')
        text = ' '.join(text.split())
        if not text:
            continue
        speaker = utterance.get('speaker_id')
        if not isinstance(speaker, str) or not speaker.strip():
            return [], 'missing_speaker'
        if speaker not in speakers:
            speakers[speaker] = len(speakers) + 1
        role = speakers[speaker]
        if turns and turns[-1][0] == role:
            turns[-1][1].append(text)
        else:
            turns.append((role, [text]))
    if not turns:
        return [], 'empty'
    if len(speakers) != 2:
        return [], 'single_speaker' if len(speakers) == 1 else 'more_than_two_speakers'
    return [(role, ' '.join(parts)) for role, parts in turns], None

def prepare_raw_pairs():
    documents, pairs, seen_ids = [], [], {}
    seen_docs, seen_pairs, skipped = set(), set(), {}
    offset = 0
    partial = raw_path.with_suffix('.partial')
    with partial.open('wb') as output:
        members = JSON_FILES
        for path in tqdm(members, desc='JSON → speaker1/speaker2 pairs'):
            member_name = path.relative_to(DATA_ROOT).as_posix()
            with path.open('r', encoding='utf-8-sig') as member:
                payload = json.load(member)
            if not isinstance(payload, dict) or not isinstance(payload.get('document'), list):
                raise ValueError(f'{member_name}: document list가 없습니다.')
            for number, document in enumerate(payload['document']):
                if not isinstance(document, dict):
                    raise ValueError(f'{member_name}: document[{number}] must be an object')
                doc_id = str(document.get('id') or f'{member_name}#{number}')
                turns, reason = dialogue_turns(document, f'{member_name}:{doc_id}')
                if reason:
                    skipped[reason] = skipped.get(reason, 0) + 1
                    continue
                doc_hash = hashlib.sha256(json.dumps(turns, ensure_ascii=False).encode()).hexdigest()
                if doc_id in seen_ids and seen_ids[doc_id] != doc_hash:
                    raise ValueError(f'Duplicate document ID with different content: {doc_id}')
                seen_ids[doc_id] = doc_hash
                if doc_hash in seen_docs:
                    skipped['duplicate_document'] = skipped.get('duplicate_document', 0) + 1
                    continue
                seen_docs.add(doc_hash)
                count = 0
                for i in range(len(turns)-1):
                    if turns[i][0] != 1 or turns[i+1][0] != 2:
                        continue
                    question, answer = turns[i][1], turns[i+1][1]
                    pair_hash = hashlib.sha256(json.dumps([question, answer], ensure_ascii=False).encode()).hexdigest()
                    if pair_hash in seen_pairs:
                        skipped['duplicate_pair'] = skipped.get('duplicate_pair', 0) + 1
                        continue
                    seen_pairs.add(pair_hash)
                    line = (json.dumps(dict(question=question, answer=answer), ensure_ascii=False) + '\n').encode('utf-8')
                    output.write(line)
                    pairs.append(dict(document=len(documents), sha256=pair_hash))
                    offset += len(line)
                    count += 1
                if count:
                    documents.append(dict(id=doc_id, member=member_name, sha256=doc_hash, pairs=count))
                else:
                    skipped['no_unique_pair'] = skipped.get('no_unique_pair', 0) + 1
    if len(documents) < 2:
        raise ValueError(f'유효한 2인 대화가 {len(documents)}개입니다. train/validation에 최소 2개 필요합니다. 제외: {skipped}')
    order = sorted(range(len(documents)), key=lambda i: hashlib.sha256(
        f'{CFG["seed"]}:{documents[i]["sha256"]}'.encode()).hexdigest())
    n_val = min(len(documents)-1, max(1, round(len(documents)*DATA_CFG['validation_fraction'])))
    validation = set(order[:n_val])
    for i, document in enumerate(documents):
        document['split'] = 'validation' if i in validation else 'train'
    for pair in pairs:
        pair['split'] = documents[pair['document']]['split']
    manifest = dict(spec=raw_spec, documents=documents, pairs=pairs, raw_bytes=offset,
                    json_files=len(members), skipped=skipped)
    partial.replace(raw_path)
    raw_index_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest

raw_spec = dict(version=3, dataset_sha256=dataset_sha256, text_field=DATA_CFG['text_field'],
                renderer='first_seen_two_speakers_adjacent_1to2_merged_turns_v1',
                seed=CFG['seed'], validation_fraction=DATA_CFG['validation_fraction'])
raw_key = hashlib.sha256(json.dumps(raw_spec, sort_keys=True).encode()).hexdigest()[:16]
raw_path = DATA_CACHE / f'{raw_key}.pairs.jsonl'
raw_index_path = DATA_CACHE / f'{raw_key}.raw.json'
assert 0 < DATA_CFG['validation_fraction'] < 1
raw_manifest = json.loads(raw_index_path.read_text(encoding='utf-8')) if raw_index_path.exists() else None
if (raw_manifest is None or raw_manifest['spec'] != raw_spec or not raw_path.exists()
        or raw_path.stat().st_size != raw_manifest['raw_bytes']):
    raw_manifest = prepare_raw_pairs()
documents, raw_pairs = raw_manifest['documents'], raw_manifest['pairs']


def tokenizer_training_indices(manifest, max_pairs, seed):
    candidates = [i for i, pair in enumerate(manifest['pairs']) if pair['split'] == 'train']
    if max_pairs is not None and (type(max_pairs) is not int or max_pairs < 1):
        raise ValueError('토크나이저 학습 쌍 수는 양의 정수 또는 None이어야 한다.')
    ordered = sorted(candidates, key=lambda i: hashlib.sha256(
        f'{seed}:{manifest["pairs"][i]["sha256"]}'.encode()).hexdigest())
    return set(ordered if max_pairs is None else ordered[:max_pairs])


def tokenizer_sentences(path, selected):
    # 선택된 train 쌍의 본문만 읽는다. validation과 화자 표식은 어휘 학습에 넣지 않는다.
    with path.open('r', encoding='utf-8') as stream:
        for i, line in enumerate(stream):
            if i in selected:
                item = json.loads(line)
                yield item['question']
                yield item['answer']


selected_tokenizer_pairs = tokenizer_training_indices(raw_manifest, TOK_CFG['max_training_pairs'], CFG['seed'])
if not selected_tokenizer_pairs:
    raise ValueError('토크나이저를 학습할 train 쌍이 없다.')
tokenizer_spec = dict(kind='sentencepiece_unigram_byte_fallback_v1', raw_spec=raw_spec,
                      options=TOK_CFG, sentencepiece_version=spm.__version__,
                      selected_pairs_sha256=hashlib.sha256(
                          json.dumps(sorted(selected_tokenizer_pairs)).encode()).hexdigest(),
                      selected_train_pairs=len(selected_tokenizer_pairs),
                      normalization='identity_no_dummy_prefix_preserve_whitespace',
                      literal_whitespace_marker='encode_U2581_as_bytes',
                      prompt_reply_boundary='encode_separately')
tokenizer_key = hashlib.sha256(json.dumps(tokenizer_spec, sort_keys=True).encode()).hexdigest()[:16]
tokenizer_dir = DATA_CACHE / 'sentencepiece' / tokenizer_key
tokenizer_dir.mkdir(parents=True, exist_ok=True)
tokenizer_path = tokenizer_dir / 'tokenizer.model'
tokenizer_manifest_path = tokenizer_dir / 'tokenizer.json'
cached_tokenizer = json.loads(tokenizer_manifest_path.read_text(encoding='utf-8')) if tokenizer_manifest_path.exists() else None
if (not tokenizer_path.exists() or cached_tokenizer is None
        or cached_tokenizer['spec'] != tokenizer_spec
        or hashlib.sha256(tokenizer_path.read_bytes()).hexdigest() != cached_tokenizer['sha256']):
    if not 512 <= TOK_CFG['vocab_size'] <= 65536:
        raise ValueError('요청 어휘 크기는 512~65536 범위여야 한다.')
    print('SentencePiece 학습: train 쌍', len(selected_tokenizer_pairs), '/ 요청 어휘', TOK_CFG['vocab_size'])
    writer = io.BytesIO()
    spm.SentencePieceTrainer.train(
        sentence_iterator=tokenizer_sentences(raw_path, selected_tokenizer_pairs), model_writer=writer,
        model_type='unigram', vocab_size=TOK_CFG['vocab_size'],
        character_coverage=TOK_CFG['character_coverage'], byte_fallback=True,
        normalization_rule_name='identity', add_dummy_prefix=False, remove_extra_whitespaces=False,
        unk_id=0, bos_id=1, eos_id=2, pad_id=3, hard_vocab_limit=False,
        input_sentence_size=0, shuffle_input_sentence=False, num_threads=1,
        max_sentence_length=TOK_CFG['max_sentence_length'], minloglevel=1)
    partial = tokenizer_path.with_suffix('.partial')
    partial.write_bytes(writer.getvalue())
    partial.replace(tokenizer_path)
    cached_tokenizer = dict(spec=tokenizer_spec, sha256=hashlib.sha256(tokenizer_path.read_bytes()).hexdigest())
    tokenizer_manifest_path.write_text(json.dumps(cached_tokenizer, ensure_ascii=False, indent=2), encoding='utf-8')
TOKENIZER = spm.SentencePieceProcessor(model_file=str(tokenizer_path))
token_meta = validate_tokenizer(TOKENIZER)
CFG.update(token_meta)
BOS, EOS, PAD, UNK, VOCAB = (token_meta[key] for key in ('bos_id', 'eos_id', 'pad_id', 'unk_id', 'vocab_size'))
tokenizer_sha256 = cached_tokenizer['sha256']
for text in ('안녕하세요? Hello 😀 𐐀', ' 화자1: 질문\n화자2: 답변 ', '실제▁문자', '<s></s><pad><unk>'):
    ids = encode_text(TOKENIZER, text)
    if UNK in ids or TOKENIZER.decode(ids) != text:
        raise ValueError(f'토크나이저 왕복 검증 실패: {text!r}')
print('SentencePiece 준비 완료:', token_meta, '모델 SHA-256:', tokenizer_sha256)


def encode(text):
    return [BOS] + encode_text(TOKENIZER, text) + [EOS]


def decode(ids):
    return TOKENIZER.decode([int(i) for i in ids if int(i) not in (BOS, EOS, PAD)])


def encode_pair(question, answer):
    # 경계 밖 병합을 막아 추론 prompt와 학습 prompt의 토큰이 정확히 같도록 한다.
    prefix = [BOS] + encode_text(TOKENIZER, render_prompt(question))
    reply = encode_text(TOKENIZER, answer) + [EOS]
    values = prefix + reply
    if UNK in values or min(values) < 0 or max(values) >= VOCAB:
        raise ValueError('인코딩 결과에 UNK 또는 잘못된 토큰 번호가 있다.')
    return np.asarray(values, dtype='<u2'), len(prefix)


data_spec = dict(version=3, raw_spec=raw_spec, tokenizer_sha256=tokenizer_sha256,
                 tokenizer=tokenizer_spec['kind'], token_meta=token_meta,
                 loss='response_and_eos_only', boundary='separate_prompt_reply')
data_key = hashlib.sha256(json.dumps(data_spec, sort_keys=True).encode()).hexdigest()[:16]
token_path = DATA_CACHE / f'{data_key}.uint16'
index_path = DATA_CACHE / f'{data_key}.json'


def prepare_token_pairs():
    pairs, offset = [], 0
    partial = token_path.with_suffix('.partial')
    with partial.open('wb') as output, raw_path.open('r', encoding='utf-8') as stream:
        for i, line in enumerate(tqdm(stream, total=len(raw_pairs), desc='SentencePiece 토큰 캐시')):
            item = json.loads(line)
            tokens, prompt_tokens = encode_pair(item['question'], item['answer'])
            tokens.tofile(output)
            pairs.append(dict(raw_pairs[i], offset=offset, length=len(tokens), prompt_tokens=prompt_tokens,
                              response_utf8_bytes=len(item['answer'].encode('utf-8'))))
            offset += len(tokens)
    if len(pairs) != len(raw_pairs):
        raise ValueError('원문 캐시와 쌍 목록의 길이가 다르다.')
    manifest = dict(spec=data_spec, documents=documents, pairs=pairs, total_tokens=offset,
                    json_files=raw_manifest['json_files'], skipped=raw_manifest['skipped'])
    partial.replace(token_path)
    index_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest


dataset_manifest = json.loads(index_path.read_text(encoding='utf-8')) if index_path.exists() else None
if (dataset_manifest is None or dataset_manifest['spec'] != data_spec or not token_path.exists()
        or token_path.stat().st_size != dataset_manifest['total_tokens'] * 2):
    dataset_manifest = prepare_token_pairs()
pairs = dataset_manifest['pairs']
tokens_mm = np.memmap(token_path, mode='r', dtype='<u2')
train_ids = [i for i, row in enumerate(pairs) if row['split'] == 'train']
val_ids = [i for i, row in enumerate(pairs) if row['split'] == 'validation']
assert train_ids and val_ids
assert selected_tokenizer_pairs.issubset(set(train_ids))
assert {pairs[i]['document'] for i in train_ids}.isdisjoint({pairs[i]['document'] for i in val_ids})
assert {pairs[i]['sha256'] for i in train_ids}.isdisjoint({pairs[i]['sha256'] for i in val_ids})
assert sum(row['length'] for row in pairs) == len(tokens_mm)
for row in pairs:
    start, end = row['offset'], row['offset'] + row['length']
    assert int(tokens_mm[start]) == BOS and int(tokens_mm[end - 1]) == EOS
    assert 1 < row['prompt_tokens'] < row['length'] - 1
data_stats = dict(json_files=dataset_manifest['json_files'], usable_documents=len(documents),
                  train_documents=len({pairs[i]['document'] for i in train_ids}),
                  validation_documents=len({pairs[i]['document'] for i in val_ids}),
                  train_pairs=len(train_ids), validation_pairs=len(val_ids), vocab_size=VOCAB,
                  tokenizer_train_pairs=len(selected_tokenizer_pairs),
                  train_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in train_ids),
                  validation_response_tokens=sum(pairs[i]['length']-pairs[i]['prompt_tokens'] for i in val_ids),
                  train_response_utf8_bytes=sum(pairs[i]['response_utf8_bytes'] for i in train_ids),
                  validation_response_utf8_bytes=sum(pairs[i]['response_utf8_bytes'] for i in val_ids),
                  excluded=dataset_manifest['skipped'])
data_stats['train_response_bytes_per_piece'] = data_stats['train_response_utf8_bytes'] / max(
    1, data_stats['train_response_tokens'] - len(train_ids))
print(json.dumps(data_stats, ensure_ascii=False, indent=2))
RUN_DIR = Path('/kaggle/working/flylm2_runs') / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR.mkdir(parents=True, exist_ok=False)
shutil.copyfile(index_path, RUN_DIR/'dataset_manifest.json')
shutil.copyfile(tokenizer_path, RUN_DIR/'tokenizer.model')
shutil.copyfile(tokenizer_manifest_path, RUN_DIR/'tokenizer.json')
(RUN_DIR/'dataset_sources.json').write_text(json.dumps(dataset_sources, ensure_ascii=False, indent=2), encoding='utf-8')
print('Kaggle 결과 저장 경로:', RUN_DIR)

## 2. 공식 원본 다운로드

MaleCNS v1.0 annotation, neuron neurotransmitter, connection Feather를 사용한다.
다운로드/길이 검사/SHA-256은 기존과 같다. 전체 annotation의 Traced neuron을 사용하고
그 뉴런들 사이에 존재하는 모든 edge/count를 보존한다. 생물학적 뉴런으로 확인되지 않은 orphan/glia/미할당 segment는 포함하지 않는다.


In [ ]:
BASE_URL = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
SOURCES = {
    'annotations': ('body-annotations-male-cns-v1.0-minconf-0.5.feather', 14483314),
    'nt': ('body-neurotransmitters-male-cns-v1.0.feather', 43282834),
    'edges': ('connectome-weights-male-cns-v1.0-minconf-0.5.feather', 1051241946),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(name, expected_size):
    target = CACHE / name
    if not target.exists() or target.stat().st_size != expected_size:
        if shutil.disk_usage(CACHE).free < expected_size + 512 * 1024**2:
            raise RuntimeError('데이터 다운로드에 필요한 디스크 공간이 부족합니다.')
        partial = target.with_suffix(target.suffix + '.part')
        with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
            with partial.open('wb') as f, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
                while True:
                    chunk = response.read(8 * 1024**2)
                    if not chunk:
                        break
                    f.write(chunk)
                    bar.update(len(chunk))
        if partial.stat().st_size != expected_size:
            raise RuntimeError(f'파일 크기 불일치: {name}. v1.0 파일/다운로드를 확인하세요.')
        partial.replace(target)
    return target

files = {key: download_file(*spec) for key, spec in SOURCES.items()}
source_manifest = {key: dict(url=BASE_URL + p.name, bytes=p.stat().st_size, sha256=sha256_file(p))
                   for key, p in files.items()}
print({key: entry['bytes'] for key, entry in source_manifest.items()})

## 3. 전체 뉴런, 전체 내부 연결의 CSR 캐시

Feather를 두 번 순회한다. 첫 순회는 destination별 edge 개수를 세고,
두 번째 순회는 디스크 memmap의 CSR 위치에 source/count를 채운다.
전체 원본 table을 pandas로 올리거나 edge 목록을 Python list에 누적하지 않는다.
중복 neuron pair는 count를 합산한다. CSR row=postsynaptic, column=presynaptic이다.
어떤 component도 버리지 않으며 self-edge도 원본에 있으면 유지한다.


In [ ]:
def require_columns(names, required, label):
    absent = set(required)-set(names)
    if absent:
        raise ValueError(f'{label}: missing {sorted(absent)}; found {list(names)}')

def map_ids(sorted_ids, values):
    positions = np.searchsorted(sorted_ids, values)
    safe = positions.clip(0, len(sorted_ids)-1)
    return positions, (positions < len(sorted_ids)) & (sorted_ids[safe] == values)

annotation = feather.read_table(files['annotations']).to_pandas()
require_columns(annotation.columns, ['bodyId','status','superclass'], 'annotations')
neurons = annotation.loc[annotation.status == 'Traced'].sort_values('bodyId').set_index('bodyId')
assert neurons.index.is_unique and not neurons.index.hasnans
body_ids = neurons.index.to_numpy(np.int64)
n = len(body_ids)
CFG['n_neurons'] = n
assert n > 100_000, f'전체 Traced 데이터인지 확인하세요: {n}'
print('Actual full Traced neuron count:', n)
print(neurons['superclass'].value_counts(dropna=False).to_string())

def full_edge_batches():
    with pa.memory_map(str(files['edges']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body_pre','body_post','weight'], 'edges')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            s, sm = map_ids(body_ids, batch.column('body_pre').to_numpy(zero_copy_only=False))
            d, dm = map_ids(body_ids, batch.column('body_post').to_numpy(zero_copy_only=False))
            keep = sm & dm
            if keep.any():
                w = batch.column('weight').to_numpy(zero_copy_only=False)[keep]
                if not np.isfinite(w).all() or (w <= 0).any():
                    raise ValueError('Invalid count')
                yield s[keep].astype(np.int32), d[keep].astype(np.int32), w.astype(np.int64)

extract_spec = dict(version=3, selection='all_status_Traced_no_component_filter',
                    orientation='post_row_pre_column', n=n,
                    source_sha256={k:v['sha256'] for k,v in source_manifest.items()})
cache_key = hashlib.sha256(json.dumps(extract_spec, sort_keys=True).encode()).hexdigest()[:16]
full_cache = CACHE / f'full_{cache_key}'
full_cache.mkdir(exist_ok=True)
manifest_path = full_cache / 'complete.json'
if not manifest_path.exists():
    degree = np.zeros(n, np.int64)
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 1: full degrees'):
        degree += np.bincount(d, minlength=n)
    raw_edges = int(degree.sum())
    if raw_edges == 0:
        raise ValueError('No full-neuron edges')
    required_disk = raw_edges*24 + 512*1024**2
    if shutil.disk_usage(full_cache).free < required_disk:
        raise RuntimeError(f'CSR 구축 디스크가 부족합니다. 약 {required_disk/2**30:.1f} GiB 여유가 필요합니다.')
    crow = np.concatenate(([0], np.cumsum(degree)))
    col = np.lib.format.open_memmap(full_cache/'src.npy', mode='w+', dtype=np.int32, shape=(raw_edges,))
    count = np.lib.format.open_memmap(full_cache/'count.npy', mode='w+', dtype=np.int64, shape=(raw_edges,))
    cursor = crow[:-1].copy()
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 2: write full CSR'):
        order = np.argsort(d, kind='stable')
        sd = d[order]
        starts = np.r_[0, np.flatnonzero(sd[1:] != sd[:-1])+1]
        sizes = np.diff(np.r_[starts, len(sd)])
        local = np.arange(len(sd))-np.repeat(starts, sizes)
        positions = cursor[sd]+local
        col[positions], count[positions] = s[order], w[order]
        cursor += np.bincount(d, minlength=n)
    assert np.array_equal(cursor, crow[1:])
    col.flush()
    count.flush()
    full = sparse.csr_matrix((count, col, crow), shape=(n,n), copy=False)
    full.sum_duplicates()
    full.sort_indices()
    nnz = int(full.nnz)
    # 중복 병합 후 유효한 배열 구간과 행 포인터를 명시적으로 저장한다.
    col[:nnz], count[:nnz] = full.indices, full.data
    col.flush()
    count.flush()
    np.save(full_cache/'crow.npy', full.indptr.astype(np.int64))
    np.save(full_cache/'body_ids.npy', body_ids)
    manifest_path.write_text(json.dumps(dict(spec=extract_spec, raw_edges=raw_edges, nnz=nnz)), encoding='utf-8')
    del full, col, count, cursor, degree, crow
    gc.collect()
full_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert full_manifest['spec'] == extract_spec
crow = np.load(full_cache/'crow.npy', mmap_mode='r')
src = np.load(full_cache/'src.npy', mmap_mode='r')[:full_manifest['nnz']]
counts = np.load(full_cache/'count.npy', mmap_mode='r')[:full_manifest['nnz']]
assert int(crow[-1]) == len(src) == len(counts)
assert np.array_equal(body_ids, np.load(full_cache/'body_ids.npy'))
graph_data = dict(body_ids=body_ids, crow=crow, src=src, counts=counts)
print(dict(neurons=n, edges=len(src), retained='all Traced-to-Traced pairs'))

## 4. 실제 sensory/motor 포트와 세포 유형

전체 Traced 집합을 유지한다. 경로가 없는 뉴런도 삭제하지 않는다.
신경전달물질은 ground truth 우선, 없으면 confidence ≥ 0.8의 예측을 사용한다.
ACh 양수 / GABA 음수는 이 모델의 근사다. 수용체 정보가 없는 다른 전달물질은 부호를 학습한다.
도파민 등을 이름만으로 보상 학습 규칙으로 해석하지 않는다.
`type` annotation이 있는 뉴런은 같은 유형끼리 동역학 매개변수를 공유하고,
미분류 뉴런은 `class`, `superclass` 순으로 묶는다. 이 fallback도 저장한다.

In [ ]:
KNOWN_NT = {'acetylcholine', 'gaba', 'glutamate', 'histamine', 'dopamine', 'serotonin', 'octopamine'}

def attach_nt(body_ids):
    selected_parts = []
    wanted = np.sort(body_ids)
    with pa.memory_map(str(files['nt']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body', 'ground_truth', 'predicted_nt',
                                              'predicted_nt_confidence', 'consensus_nt'], 'neurotransmitters')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            _, keep = map_ids(wanted, batch.column('body').to_numpy(zero_copy_only=False))
            if keep.any():
                selected_parts.append(batch.filter(pa.array(keep)).to_pandas())
    if not selected_parts:
        raise RuntimeError('선택된 body와 전달물질 테이블의 join이 비어 있습니다.')
    table = pd.concat(selected_parts, ignore_index=True)
    if table['body'].duplicated().any():
        raise ValueError('Duplicate NT body IDs: explicit aggregation is required')
    table = table.set_index('body').reindex(body_ids)
    truth = table['ground_truth'].fillna('').str.strip().str.lower()
    predicted = table['predicted_nt'].fillna('').str.strip().str.lower()
    confidence = pd.to_numeric(table['predicted_nt_confidence'], errors='coerce')
    trusted = truth.isin(KNOWN_NT)
    confident = predicted.isin(KNOWN_NT) & confidence.ge(CFG['nt_confidence']) & confidence.le(1)
    effective = np.where(trusted, truth, np.where(confident, predicted, 'unknown'))
    origin = np.where(trusted, 'ground_truth', np.where(confident, 'predicted_nt', 'unknown'))
    table['effective_nt'] = effective
    table['effective_source'] = origin
    table.index.name = 'body'
    return table


SENSORY_CLASSES = ['cb_sensory','ol_sensory','vnc_sensory','sensory_ascending','sensory_descending']
MOTOR_CLASSES = ['cb_motor','vnc_motor']
input_nodes = np.flatnonzero(neurons.superclass.isin(SENSORY_CLASSES)).astype(np.int64)
output_nodes = np.flatnonzero(neurons.superclass.isin(MOTOR_CLASSES)).astype(np.int64)
assert len(input_nodes) and len(output_nodes)
assert len(np.intersect1d(input_nodes, output_nodes)) == 0
nt_table = attach_nt(body_ids)
node_sign = nt_table.effective_nt.map({'acetylcholine':1.0,'gaba':-1.0}).fillna(0).to_numpy(np.float32)
if not CFG['use_nt_signs']:
    node_sign[:] = 0
edge_sign = node_sign[src]
from scipy.sparse.csgraph import dijkstra
# 다중 시작점의 최소 거리만 구해 뉴런 수 길이의 벡터를 유지한다.
adj_post_pre = sparse.csr_matrix((np.ones(len(src), np.uint8), src, crow), shape=(n,n), copy=False)
adj_pre_post = adj_post_pre.T.tocsr()
from_sensory = dijkstra(adj_pre_post, directed=True, indices=input_nodes, min_only=True, unweighted=True)
to_motor = dijkstra(adj_post_pre, directed=True, indices=output_nodes, min_only=True, unweighted=True)
graph_stats = dict(neurons=n, edges=len(src), sensory_neurons=len(input_nodes), motor_neurons=len(output_nodes),
                   reachable_from_sensory=int(np.isfinite(from_sensory).sum()),
                   can_reach_motor=int(np.isfinite(to_motor).sum()),
                   sensory_to_motor_path_neurons=int((np.isfinite(from_sensory)&np.isfinite(to_motor)).sum()),
                   reachable_motors=int(np.isfinite(from_sensory[output_nodes]).sum()),
                   positive_edges=int((edge_sign>0).sum()), negative_edges=int((edge_sign<0).sum()),
                   unconstrained_edges=int((edge_sign==0).sum()))
assert graph_stats['reachable_motors'] > 0, 'No biological sensory→motor path; annotations/direction를 확인하세요.'
print(graph_stats)
print(nt_table.groupby(['effective_source','effective_nt']).size().to_string())
neurons[['superclass','class']].assign(is_input=neurons.index.isin(body_ids[input_nodes]),
               is_output=neurons.index.isin(body_ids[output_nodes])).to_csv(RUN_DIR/'neuron_ports.csv')
nt_table.to_csv(RUN_DIR/'neurotransmitters.csv')
del adj_post_pre, adj_pre_post, annotation
gc.collect()


def dynamics_types(frame):
    labels = pd.Series('', index=frame.index, dtype=object)
    for column in ['type', 'class', 'superclass']:
        if column not in frame:
            continue
        values = frame[column].fillna('').astype(str).str.strip()
        valid = ~values.str.lower().isin(['', 'none', 'nan', 'unknown', 'tbc', 'untyped'])
        use = labels.eq('') & valid
        labels.loc[use] = column + ':' + values.loc[use]
    labels = labels.mask(labels.eq(''), 'unknown')
    codes, names = pd.factorize(labels, sort=True)
    return codes.astype(np.int64), list(names)

type_ids, type_names = dynamics_types(neurons)
print('Dynamics groups:', len(type_names), 'type-annotated neurons:',
      sum(type_names[i].startswith('type:') for i in type_ids))

## 4A. 문헌 근거와 미확인 유형의 명시적 기본값

MaleCNS 자체에는 완전한 발화 방식 표가 없다. 아래 근거집은 `type`의 정확한 문자열만 매칭한다.
부분 문자열·정규식·전달물질·상위 class를 이용해 미확인 유형에 생리학 라벨을 퍼뜨리지 않는다.
다른 개체의 세포 유형을 대응시키는 단계는 추론이며, MaleCNS 개체를 직접 기록한 것이 아니다.

| MaleCNS 유형 | 모델 방식 | 근거와 한계 |
|---|---|---|
| H2 | 스파이크 | [Zhao 외, 2025](https://doi.org/10.1038/s41586-025-09276-5)의 H2 전기생리 기록. 개체 간 유형 전이. |
| DM1_lPN | 스파이크 | [Gouwens·Wilson, 2009](https://doi.org/10.1523/JNEUROSCI.0764-09.2009)의 DM1 PN 활동전위. 암컷 → 수컷 및 해부학적 이름 대응 가정. |
| HSN, HSE, HSS | 연속 | [Pokusaeva 외, 2024](https://doi.org/10.1038/s41467-024-53173-w)의 성체 수컷 HS 막전위 기록. 전기 시냅스 효과까지 재현하지 않는다. |
| R1-R6 | 연속 | [Juusola 외, 2016](https://doi.org/10.3791/54142)의 광수용체 전압 기록. 통합된 세포군 수준의 대응. |
| 나머지 | 미확인 → 연속 기본값 | 문헌에서 비스파이킹으로 확인된 것이 아니다. |

주된 신호 방식만 단순화한다. 스파이크 뉴런도 역치 아래에서는 연속 막전위를 가지며,
연속 모델로 분류한 세포도 특정 조건의 재생성 사건을 모두 배제했다는 뜻은 아니다.
근거는 방식 선택에만 쓰고 시간 상수·역치·불응기 수치는 생리학 측정값으로 주장하지 않는다.
근거 표를 수정하면 새 학습을 시작해야 한다. 현재 실행의 매핑과 근거는 결과 폴더와 추론 번들에 고정 저장된다.


In [ ]:
# 문헌의 세포 유형을 MaleCNS의 정확한 type 문자열에 연결하는 작은 근거집이다.
# 이 표는 MaleCNS 원본의 생리학 라벨도, 모든 세포 유형의 문헌 조사 결과도 아니다.
EVIDENCE_VERSION = 'flylm2_activity_evidence_2026-09-08_v1'
ACTIVITY_EVIDENCE = [
    dict(type_name='H2', activity='spiking',
         evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
         preparation='성체; H2 전세포 기록', sex='이 근거집에서는 성별을 확정하지 않음',
         source='https://doi.org/10.1038/s41586-025-09276-5',
         locator='H2 electrophysiology; 국소 운동 자극에 대한 스파이크 기록',
         finding='H2의 활동전위를 직접 기록하였다.',
         mapping='MaleCNS type H2와 문헌의 H2를 세포 유형 수준에서 대응시킨다.',
         limitation='다른 개체의 유형 근거이며 MaleCNS 개체의 직접 측정이 아니다. 역치·불응기는 옮겨오지 않는다.'),
    dict(type_name='DM1_lPN', activity='spiking',
         evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
         preparation='성체 2–10일; DM1 투사뉴런 전세포 기록', sex='female',
         source='https://doi.org/10.1523/JNEUROSCI.0764-09.2009',
         locator='Methods: NP5221-Gal4 DM1 PNs; Fig. 5B',
         finding='DM1 투사뉴런에서 실제 활동전위를 기록하였다.',
         mapping='DM1 uniglomerular PN을 MaleCNS DM1_lPN에 대응시키는 해부학적 추론이다.',
         limitation='암컷의 DM1 유형 근거를 수컷 대응 유형으로 전이한다. 모든 후각 투사뉴런으로 확장하지 않는다.'),
    *[dict(type_name=name, activity='graded',
           evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
           preparation='성체 수컷; 생체 내 HS 전세포 기록', sex='male',
           source='https://doi.org/10.1038/s41467-024-53173-w',
           locator='Introduction; Fig. 5B–D; Methods: male flies',
           finding='HS의 운동 반응은 연속 막전위 변화로 다룬다. HSN/HSE/HSS 기록을 구별한다.',
           mapping=f'MaleCNS type {name}를 문헌의 동명 HS 유형에만 대응시킨다.',
           limitation='주된 연속 신호를 근사한다. 모든 조건에서 재생성 사건이 불가능하다는 주장이 아니다. 전기 시냅스는 추가하지 않는다.')
      for name in ('HSN', 'HSE', 'HSS')],
    dict(type_name='R1-R6', activity='graded',
         evidence_level='recorded_family_transfer', species='Drosophila melanogaster',
         preparation='생체 내 R1–R6 광수용체 세포 내 전압 기록', sex='이 근거집에서는 성별을 확정하지 않음',
         source='https://doi.org/10.3791/54142',
         locator='Introduction; Representative Results: R1–R6 voltage responses',
         finding='R1–R6의 빛 반응을 연속 전압 신호로 기록한다.',
         mapping='MaleCNS의 통합 라벨 R1-R6에만 적용한다. R7/R8에는 전파하지 않는다.',
         limitation='기록된 세포군의 주된 신호 방식을 전이한다. 개별 R1–R6의 매개변수 차이는 보존하지 못한다.'),
]


def assign_activity_modes(frame, evidence, unknown_mode='graded'):
    if unknown_mode != 'graded':
        raise ValueError('미확인 뉴런의 기본값은 이 버전에서 graded만 허용한다.')
    if not frame.index.is_unique or 'type' not in frame:
        raise ValueError('고유 bodyId 인덱스와 type 열이 필요하다.')
    lookup = {}
    for entry in evidence:
        name = entry['type_name']
        if not isinstance(name, str) or not name or name != name.strip() or name in lookup:
            raise ValueError(f'중복 또는 잘못된 근거 유형: {name}')
        if entry['activity'] not in ('graded', 'spiking'):
            raise ValueError(f'잘못된 신호 방식: {name}')
        for key in ('source', 'locator', 'finding', 'mapping', 'limitation',
                    'evidence_level', 'species', 'preparation', 'sex'):
            if not isinstance(entry.get(key), str) or not entry[key].strip():
                raise ValueError(f'근거 필드 누락: {name}, {key}')
        if not entry['source'].startswith('https://'):
            raise ValueError(f'근거 출처에는 HTTPS 링크가 필요하다: {name}')
        lookup[name] = entry
    types = frame['type'].fillna('').astype(str).str.strip()
    # 0은 미확인, 1은 문헌에서 연속 신호를 지지, 2는 문헌에서 스파이크를 지지한다.
    codes = types.map({k: 2 if v['activity'] == 'spiking' else 1 for k, v in lookup.items()})
    codes = codes.fillna(0).to_numpy(dtype=np.uint8)
    records = pd.DataFrame({'bodyId': frame.index.to_numpy(), 'type': types.to_numpy(),
                            'evidence_code': codes})
    records['evidence_activity'] = np.take(['unknown', 'graded', 'spiking'], codes)
    records['assigned_activity'] = np.where(codes == 2, 'spiking', unknown_mode)
    for key in ('source', 'locator', 'finding', 'mapping', 'limitation',
                'evidence_level', 'species', 'preparation', 'sex'):
        records[key] = types.map({k: v[key] for k, v in lookup.items()}).fillna('').to_numpy()
    records.loc[codes == 0, 'limitation'] = '이 근거집에서 미확인; 연속 모델을 공학적 기본값으로 사용한다.'
    records.loc[codes == 0, 'evidence_level'] = 'unknown_default'
    by_type = records.groupby(['type', 'evidence_activity', 'assigned_activity'], dropna=False).size()
    summary = dict(neurons=len(frame), evidence_supported_neurons=int((codes != 0).sum()),
                   evidence_spiking_neurons=int((codes == 2).sum()),
                   evidence_graded_neurons=int((codes == 1).sum()),
                   unknown_default_neurons=int((codes == 0).sum()),
                   evidence_fraction=float((codes != 0).mean()),
                   unmatched_evidence_types=sorted(set(lookup) - set(types)),
                   unknown_default=unknown_mode)
    return codes, records, by_type.reset_index(name='neurons'), summary


dynamics_ids, activity_annotations, activity_types, activity_coverage = assign_activity_modes(
    neurons, ACTIVITY_EVIDENCE, CFG['unknown_activity'])
activity_annotations['effective_activity'] = np.where(
    (dynamics_ids == 2) & (CFG['neuron_mode'] == 'mixed'), 'spiking', 'graded')
activity_coverage['effective_spiking_neurons'] = int(
    ((dynamics_ids == 2) & (CFG['neuron_mode'] == 'mixed')).sum())
activity_coverage['neuron_mode'] = CFG['neuron_mode']
dynamics_spec = dict(version=EVIDENCE_VERSION, mode=CFG['neuron_mode'],
                     unknown_activity=CFG['unknown_activity'], evidence=ACTIVITY_EVIDENCE,
                     state_layout=['membrane', 'synaptic_signal', 'adaptation', 'refractory_ticks'],
                     time_unit='simulation_tick_not_ms', parameters='engineering_assumptions')
dynamics_json = json.dumps(dynamics_spec, ensure_ascii=False, sort_keys=True, indent=2)
dynamics_sha256 = hashlib.sha256(dynamics_json.encode('utf-8')).hexdigest()
activity_annotations.to_csv(RUN_DIR/'neuron_activity_annotations.csv', index=False)
activity_types.to_csv(RUN_DIR/'activity_types.csv', index=False)
(RUN_DIR/'dynamics_evidence.json').write_bytes(dynamics_json.encode('utf-8'))
(RUN_DIR/'activity_coverage.json').write_text(
    json.dumps(activity_coverage, ensure_ascii=False, indent=2), encoding='utf-8')
print('신경 활동 근거 범위:', json.dumps(activity_coverage, ensure_ascii=False, indent=2))
print(activity_types.loc[activity_types.evidence_activity != 'unknown'].to_string(index=False))
if activity_coverage['unmatched_evidence_types']:
    print('주의: 원본 type과 일치하지 않아 적용되지 않은 근거:', activity_coverage['unmatched_evidence_types'])
if CFG['neuron_mode'] == 'mixed' and not np.any(dynamics_ids == 2):
    raise ValueError('스파이크 근거가 하나도 매칭되지 않았다. type 이름/원본 버전을 확인하세요.')
print('미확인은 비스파이킹이 확인되었다는 뜻이 아니다. 현재 연속 기본값을 사용한다.')

## 5. 유형별 연속/스파이크 동역학과 내부 상태

각 tick에서 이전 `s`가 원본 연결망을 따라 전류를 전달한다. 그 다음 막 상태를 적분한다.
- 연속 유형과 미확인 기본값: `r = relu(tanh(v − threshold))`. v1의 갱신식을 유지한다.
- 스파이크 유형: 역치 이상이면 `z=1`, 아니면 `z=0`; 발화하면 막전위를 리셋하고 2 tick 동안 재발화를 막는다.
- 불응기 중에도 시냅스 신호와 적응은 감쇠하지만 막전위는 리셋 값에 고정한다.
- `s`와 `a`는 각각 `r` 또는 `z`를 빠르게/느리게 추적한다. 운동 출력은 필터된 `s`를 읽는다.

순전파의 스파이크는 실제 이산 사건이다. 학습에는 빠른 시그모이드 대리 미분을 사용하며,
리셋 여부와 불응기 카운터는 미분하지 않는다. 이는 생물학적 학습 규칙이 아니다.
기존 유형별 시간 상수와 적응 강도, 시냅스 수 prior, 알려진 전달물질 부호 제약을 유지한다.
초기 상태는 모두 0이고, padding 동안 네 상태 모두 멈춘다. 배치 사이에는 상태를 초기화한다.
현재 CSR 연산은 매 tick 연결을 처리하므로 스파이크가 희소해도 GPU 속도 향상을 보장하지 않는다.

In [ ]:
class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, voltage_gap, slope):
        ctx.save_for_backward(voltage_gap)
        ctx.slope = slope
        return (voltage_gap >= 0).to(voltage_gap.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (gap,) = ctx.saved_tensors
        # 순전파는 이산 사건이며, 역전파에만 빠른 시그모이드의 대리 미분을 쓴다.
        return grad_output / (1 + ctx.slope * gap.abs()).square(), None


def mixed_neuron_tick(v, signal, slow, refractory, current, alpha, threshold,
                      adaptation, spiking_mask, reset_voltage, refractory_ticks, surrogate_slope):
    candidate = v + alpha[0] * (-v + current - adaptation * slow)
    blocked = spiking_mask & (refractory > 0)
    candidate = torch.where(blocked, torch.full_like(candidate, reset_voltage), candidate)
    spikes = SurrogateSpike.apply(candidate - threshold, surrogate_slope)
    spikes = spikes * (spiking_mask & ~blocked).to(candidate.dtype)
    graded = torch.relu(torch.tanh(candidate - threshold))
    release = torch.where(spiking_mask, spikes, graded)
    # 리셋·불응기 사건의 이산 결정에는 별도 대리 미분을 적용하지 않는다.
    fired = spikes.detach() > 0
    v = torch.where(fired, torch.full_like(candidate, reset_voltage), candidate)
    refractory = torch.where(fired, torch.full_like(refractory, float(refractory_ticks)),
                             torch.clamp(refractory - 1, min=0))
    refractory = torch.where(spiking_mask, refractory, torch.zeros_like(refractory)).detach()
    signal = signal + alpha[1] * (release - signal)
    slow = slow + alpha[2] * (release - slow)
    return v, signal, slow, refractory, spikes


def validate_dynamics_config(cfg):
    if cfg['neuron_mode'] not in ('mixed', 'graded_control'):
        raise ValueError('neuron_mode은 mixed 또는 graded_control이어야 한다.')
    if cfg['unknown_activity'] not in ('graded', 'spiking'):
        raise ValueError('미확인 유형의 기본 동역학은 graded 또는 spiking이어야 한다.')
    for key in ('spike_threshold_center', 'spike_threshold_radius', 'spike_reset', 'surrogate_slope'):
        if not math.isfinite(cfg[key]):
            raise ValueError(f'유한한 동역학 설정이 필요하다: {key}')
    if (cfg['spike_threshold_radius'] < 0 or cfg['surrogate_slope'] <= 0
            or cfg['spike_threshold_center'] - cfg['spike_threshold_radius'] <= max(0, cfg['spike_reset'])):
        raise ValueError('스파이크 역치는 항상 0과 리셋 전압보다 높아야 한다.')
    if (type(cfg['refractory_ticks']) is not int or cfg['refractory_ticks'] < 0
            or type(cfg['brain_steps']) is not int or cfg['brain_steps'] < 1):
        raise ValueError('불응기는 0 이상의 정수, 토큰당 tick 수는 양의 정수여야 한다.')


class ConnectomeLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, dynamics_ids, device):
        super().__init__()
        validate_dynamics_config(cfg)
        self.tbptt = cfg['tbptt']
        self.vocab_size = cfg['vocab_size']
        self.bos_id, self.eos_id = cfg['bos_id'], cfg['eos_id']
        self.pad_id, self.unk_id = cfg['pad_id'], cfg['unk_id']
        self.tokenizer = None
        self.reset_voltage = cfg['spike_reset']
        self.refractory_ticks = cfg['refractory_ticks']
        self.surrogate_slope = cfg['surrogate_slope']
        self.spike_threshold_center = cfg['spike_threshold_center']
        self.spike_threshold_radius = cfg['spike_threshold_radius']
        codes = np.asarray(dynamics_ids)
        if (codes.shape != (cfg['n_neurons'],) or codes.dtype != np.uint8
                or not np.isin(codes, [0, 1, 2]).all()):
            raise ValueError('뉴런 순서에 맞는 uint8 활동 코드 0/1/2가 필요하다.')
        mask = (codes == 2) & (cfg['neuron_mode'] == 'mixed')
        self.register_buffer('spiking_mask', torch.tensor(mask, dtype=torch.bool, device=device), persistent=False)
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        d = cfg['embedding_dim']
        self.embedding = nn.Embedding(self.vocab_size, d, device=device)
        self.input_adapter = nn.Linear(d, len(inputs), bias=False, device=device)
        self.motor_norm = nn.LayerNorm(len(outputs), eps=1e-6, elementwise_affine=False, device=device)
        self.output_adapter = nn.Linear(len(outputs), d, bias=False, device=device)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        offset = self.threshold.tanh()[self.type_ids]
        threshold = torch.where(self.spiking_mask,
            self.spike_threshold_center + self.spike_threshold_radius * offset, 0.5 * offset)
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return self.embedding.weight.new_zeros(4, batch, self.n)

    def advance(self, token, state, weights, alpha, threshold, adaptation, collect_spikes=False):
        v, signal, slow, refractory = state.unbind(0)
        value = self.input_adapter(self.embedding(token))
        drive = torch.zeros_like(v).index_add(1, self.inputs, value)
        counts = torch.zeros_like(v) if collect_spikes else None
        for _ in range(self.k):
            message = SparseEdges.apply(signal, weights, self.rowptr, self.src, self.dst, self.block)
            v, signal, slow, refractory, spikes = mixed_neuron_tick(
                v, signal, slow, refractory, message + drive, alpha, threshold, adaptation,
                self.spiking_mask, self.reset_voltage, self.refractory_ticks, self.surrogate_slope)
            if collect_spikes:
                counts = counts + spikes.detach()
        result = torch.stack((v, signal, slow, refractory))
        return (result, counts) if collect_spikes else result

    def forward(self, tokens, state=None, active_mask=None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError('Nonempty tokens [B,T] required')
        state = self.initial_state(tokens.shape[0]) if state is None else state
        if state.shape != (4, tokens.shape[0], self.n):
            raise ValueError('상태는 [4,B,N]: 막전위, 시냅스 신호, 적응, 남은 불응기여야 한다.')
        if active_mask is not None and (active_mask.shape != tokens.shape or active_mask.dtype != torch.bool):
            raise ValueError('active_mask must be bool [B,T]')
        weights = self.effective_weights()
        alpha, threshold, adaptation = self.dynamics()
        logits = []
        for t in range(tokens.shape[1]):
            previous = state
            args = (tokens[:, t], state, weights, alpha, threshold, adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                state = checkpoint(self.advance, *args, use_reentrant=False, preserve_rng_state=False)
            else:
                state = self.advance(*args)
            if active_mask is not None:
                state = torch.where(active_mask[:, t][None, :, None], state, previous)
            motor = self.motor_norm(state[1].index_select(1, self.outputs))
            logits.append(F.linear(self.output_adapter(motor), self.embedding.weight))
        return torch.stack(logits, dim=1), state

def make_model():
    torch.manual_seed(CFG['seed'])
    torch.cuda.manual_seed_all(CFG['seed'])
    model = ConnectomeLM(CFG, crow, src, counts, edge_sign, input_nodes, output_nodes, type_ids, dynamics_ids, device)
    model.tokenizer = TOKENIZER
    return model

# 뉴런당 단일 구획 근사를 유지한다.

## 6. 혼합 동역학 검증과 전체 그래프 메모리 검사

작은 수치 검사에서 이산 발화·리셋·불응기·정지 입력·대리 미분과 기존 연속식의 일치를 확인한다.
희소 연결의 순전파/역전파는 작은 밀집 행렬과 비교한다.
그 다음 **전체 MaleCNS + 64토큰 + backward + Adam**을 검사한다.
이 메모리 검사의 토큰은 합성 입력이며 학습 데이터에 추가되지 않는다. 시험 모델은 버린다.
메모리 부족 또는 수치 문제가 있으면 학습 전에 중단한다. 뉴런이나 연결을 줄이지 않는다.

In [ ]:
def check_mixed_dynamics(test_device):
    # 전체 그래프 학습 전에 리셋·불응기·연속 대조군·대리 미분을 검사한다.
    dtype = torch.float64
    alpha = torch.ones(3, 2, dtype=dtype, device=test_device)
    zeros = torch.zeros(1, 2, dtype=dtype, device=test_device)
    mask = torch.tensor([True, False], device=test_device)
    threshold = torch.tensor([0.5, 0.0], dtype=dtype, device=test_device)
    state = [zeros.clone() for _ in range(4)]
    events = []
    for _ in range(7):
        *state, z = mixed_neuron_tick(*state, torch.ones_like(zeros), alpha, threshold,
                                     zeros[0], mask, 0., 2, 5.)
        events.append(int(z[0, 0]))
        assert z[0, 1] == 0 and state[3][0, 1] == 0
        torch.testing.assert_close(state[1][0, 1], torch.tanh(zeros.new_tensor(1.)))
    assert events == [1, 0, 0, 1, 0, 0, 1], events
    assert state[0][0, 0] == 0 and state[3][0, 0] == 2
    silent = mixed_neuron_tick(*[zeros.clone() for _ in range(4)], zeros, alpha,
                               threshold, zeros[0], mask, 0., 2, 5.)
    assert silent[-1].sum() == 0
    gap = torch.tensor([-0.1, 0.0, 0.1], dtype=dtype, device=test_device, requires_grad=True)
    binary = SurrogateSpike.apply(gap, 5.)
    torch.testing.assert_close(binary, gap.new_tensor([0., 1., 1.]))
    binary.sum().backward()
    assert torch.isfinite(gap.grad).all() and (gap.grad > 0).all()

    # 스파이크를 사용하지 않는 경우 v1의 연속 갱신식과 수치적으로 일치한다.
    v, s, a = [zeros + x for x in (0.1, 0.2, 0.3)]
    coeff = alpha * 0.25
    current = zeros + 0.7
    adapt = zeros[0] + 0.1
    old_v = v + coeff[0] * (-v + current - adapt * a)
    old_r = torch.relu(torch.tanh(old_v - threshold))
    expected = [old_v, s + coeff[1] * (old_r - s), a + coeff[2] * (old_r - a)]
    actual = mixed_neuron_tick(v, s, a, zeros, current, coeff, threshold, adapt,
                               torch.zeros_like(mask), 0., 2, 5.)
    for got, want in zip(actual[:3], expected):
        torch.testing.assert_close(got, want, rtol=0, atol=0)

    # 희소 연결의 미분은 작은 밀집 행렬 계산과 비교한다.
    rowptr = torch.tensor([0, 2, 3], device=test_device)
    col = torch.tensor([0, 1, 0], device=test_device)
    dst = torch.tensor([0, 0, 1], device=test_device)
    h = torch.tensor([[0.2, 0.4], [0.5, -0.1]], dtype=dtype, device=test_device, requires_grad=True)
    w = torch.tensor([0.3, -0.2, 0.8], dtype=dtype, device=test_device, requires_grad=True)
    out = SparseEdges.apply(h, w, rowptr, col, dst, 2)
    dense = torch.stack((w[:2], torch.stack((w[2], w[2] * 0))))
    ref = h @ dense.T
    torch.testing.assert_close(out, ref)
    gradients = torch.autograd.grad(out.square().sum(), (h, w), retain_graph=True)
    reference = torch.autograd.grad(ref.square().sum(), (h, w))
    for got, want in zip(gradients, reference):
        torch.testing.assert_close(got, want)
    print('통과: 스파이크 발생·리셋·불응기·연속식 일치·대리 미분·희소 연결 미분')


check_mixed_dynamics(device)


def check_full_graph_memory(batch):
    # 토큰 내용 대신 64토큰 길이의 최악 메모리 경로를 확인하는 합성 입력이다.
    trial = trial_optimizer = ids = logits = state = loss = None
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    budget = min(free, total) * CFG['vram_fraction']
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    row = dict(batch_size=batch, budget_GiB=budget / 2**30, fits=False)
    try:
        trial = make_model()
        trial_optimizer = torch.optim.Adam(trial.parameters(), lr=CFG['lr'], foreach=False)
        ids = torch.arange(CFG['tbptt'], device=device).remainder(VOCAB)[None].repeat(batch, 1)
        logits, state = trial(ids)
        loss = F.cross_entropy(logits.reshape(-1, VOCAB), ids.roll(-1, dims=1).reshape(-1))
        if not torch.isfinite(loss):
            raise RuntimeError('메모리 검사에서 유한하지 않은 손실을 발견했다.')
        loss.backward()
        nn.utils.clip_grad_norm_(trial.parameters(), 1.0, error_if_nonfinite=True)
        trial_optimizer.step()
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_reserved()
        row.update(peak_reserved_GiB=peak / 2**30, fits=peak <= budget)
    except torch.cuda.OutOfMemoryError:
        row['reason'] = 'CUDA 메모리 부족'
    finally:
        del trial, trial_optimizer, ids, logits, state, loss
        gc.collect()
        torch.cuda.empty_cache()
    row['seconds'] = time.perf_counter() - started
    return row


batch_checks = []
if CFG['batch_size'] == 'auto':
    limit = min(len(train_ids), CFG['max_batch_size'] or len(train_ids))
    candidate, chosen = 1, None
    while candidate <= limit:
        row = check_full_graph_memory(candidate)
        batch_checks.append(row)
        print('전체 그래프 메모리 검사:', row)
        if not row['fits']:
            break
        chosen = candidate
        if candidate == limit:
            break
        candidate = min(candidate * 2, limit)
    if chosen is None:
        raise RuntimeError('배치 1도 메모리 예산에 맞지 않는다. 그래프를 줄이지 않고 중단한다.')
    CFG['batch_size'] = chosen
else:
    row = check_full_graph_memory(CFG['batch_size'])
    batch_checks.append(row)
    print('전체 그래프 메모리 검사:', row)
    if not row['fits']:
        raise RuntimeError('지정한 배치가 메모리 예산을 넘었다. batch_size를 줄이거나 auto로 설정하세요.')
(RUN_DIR/'batch_tuning.json').write_text(json.dumps(batch_checks, ensure_ascii=False, indent=2), encoding='utf-8')
print('선택된 학습 배치:', CFG['batch_size'])

## 7. 실제 다중 배치 학습

비슷한 prompt/답변 길이끼리 배치를 구성해 padding 낭비를 줄인다.
각 대화의 prompt 앞부분은 독립적으로 warmup하고, 마지막 최대 32 prompt tokens는 답변과 함께 역전파한다.
padding 동안 해당 행의 신경 상태는 멈추고 loss에서도 제외한다. 배치가 바뀔 때 모든 행의 상태를 초기화한다.
loss는 padding을 제외한 전체 답변 SentencePiece 토큰 수로 나눈다. TBPTT는 각 대화별 64단계로 유지한다.
batch update 수와 `response_tokens_per_second`를 함께 출력한다.
checkpoint는 추론/추가 학습용 가중치이며 optimizer/data cursor의 정확한 재개는 제공하지 않는다.

`response_tokens_per_second`와 perplexity의 단위도 SentencePiece 토큰이다. 바이트 버전 수치와 직접 비교하지 않는다.
전처리 로그의 `train_response_bytes_per_piece`는 EOS를 제외한 토큰당 원문 UTF-8 바이트 수이다.
토큰 수가 줄어도 언어 품질이 좋아졌다는 뜻은 아니며, 같은 텍스트와 계산 예산에서 별도로 평가해야 한다.

In [ ]:
IGNORE_INDEX = -100
assert (BOS, EOS, PAD, UNK) == (1, 2, 3, 0)

def response_mask(start, length, prompt_tokens):
    # x[k]가 예측하는 y[k]의 원래 sequence 위치는 start+k+1이다.
    return np.arange(start+1, start+length+1) >= prompt_tokens

def chunk_spans(length, prompt_tokens):
    warmup = max(0, prompt_tokens - 1 - CFG['prompt_grad_tokens'])
    for start in range(0, warmup, CFG['tbptt']):
        yield start, min(CFG['tbptt'], warmup - start)
    for start in range(warmup, length - 1, CFG['tbptt']):
        yield start, min(CFG['tbptt'], length - 1 - start)

def batched_chunks(indices, max_response_tokens=None):
    rows = [pairs[i] for i in indices]
    warmups = [max(0, row['prompt_tokens'] - 1 - CFG['prompt_grad_tokens']) for row in rows]
    stops = [row['length'] if max_response_tokens is None else
             min(row['length'], row['prompt_tokens'] + max_response_tokens) for row in rows]
    # 각 행의 준비 입력을 먼저 읽고, 짧은 행은 자신의 경계에서 상태를 멈춘다.
    for warmup_phase in (True, False):
        starts = [0] * len(rows) if warmup_phase else warmups
        lengths = warmups if warmup_phase else [stop - 1 - start for stop, start in zip(stops, starts)]
        for offset in range(0, max(lengths), CFG['tbptt']):
            width = min(CFG['tbptt'], max(lengths) - offset)
            x = np.full((len(rows), width), PAD, dtype=np.int64)
            y = np.full_like(x, IGNORE_INDEX)
            active = np.zeros(x.shape, dtype=bool)
            counts = np.zeros(len(rows), dtype=np.int64)
            for lane, row in enumerate(rows):
                take = min(width, max(0, lengths[lane] - offset))
                if take == 0:
                    continue
                start = starts[lane] + offset
                begin = row['offset'] + start
                values = tokens_mm[begin:begin+take+1]
                x[lane, :take] = values[:-1]
                active[lane, :take] = True
                if not warmup_phase:
                    mask = response_mask(start, take, row['prompt_tokens'])
                    # -100을 넣기 전에 uint16 토큰을 부호 있는 int64로 복사한다.
                    y[lane, :take] = values[1:]
                    y[lane, :take][~mask] = IGNORE_INDEX
                    counts[lane] = int(mask.sum())
            # CUDA에 전달하기 전에 CPU에서 잘못된 토큰 번호를 검사한다.
            invalid_x = (x < 0) | (x >= VOCAB)
            invalid_y = (y != IGNORE_INDEX) & ((y < 0) | (y >= VOCAB))
            if invalid_x.any() or invalid_y.any():
                raise ValueError(f'Invalid token IDs in pairs {indices}: '
                                 f'inputs={x[invalid_x][:8].tolist()}, labels={y[invalid_y][:8].tolist()}')
            yield (torch.from_numpy(x).to(device), torch.from_numpy(y).to(device),
                   torch.from_numpy(active).to(device), counts)

def epoch_batches(epoch):
    rng = random.Random(CFG['seed'] + epoch)
    order = train_ids.copy()
    rng.shuffle(order)
    batch = CFG['batch_size']
    groups = []
    def length_key(index):
        row = pairs[index]
        warmup = max(0, row['prompt_tokens'] - 1 - CFG['prompt_grad_tokens'])
        return (warmup // CFG['tbptt'], (row['length'] - warmup) // CFG['tbptt'])
    for start in range(0, len(order), batch * 32):
        pool = sorted(order[start:start+batch*32], key=length_key)
        groups.extend(pool[i:i+batch] for i in range(0, len(pool), batch))
    rng.shuffle(groups)
    return groups

def batch_updates(indices):
    return max(math.ceil((pairs[i]['length'] - 1 - max(
        0, pairs[i]['prompt_tokens'] - 1 - CFG['prompt_grad_tokens'])) / CFG['tbptt']) for i in indices)

def check_response_format():
    fixture = dict(utterance=[
        dict(speaker_id='A', form='질문 하나'), dict(speaker_id='A', form='질문 둘'),
        dict(speaker_id='B', form='답변 하나'), dict(speaker_id='A', form='다음 질문'),
        dict(speaker_id='B', form='다음 답변')])
    # text_field 설정과 무관하게 동일 구조를 검사한다.
    for utterance in fixture['utterance']:
        utterance[DATA_CFG['text_field']] = utterance['form']
    turns, reason = dialogue_turns(fixture, 'fixture')
    assert reason is None and turns == [(1, '질문 하나 질문 둘'), (2, '답변 하나'), (1, '다음 질문'), (2, '다음 답변')]
    assert dialogue_turns(dict(utterance=fixture['utterance'][:2]), 'monologue')[1] == 'single_speaker'
    encoded, boundary = encode_pair('질문', '답변')
    actual_targets = []
    for start, length in chunk_spans(len(encoded), boundary):
        mask = response_mask(start, length, boundary)
        actual_targets.extend(encoded[start+1:start+length+1][mask].tolist())
    assert actual_targets == encode_text(TOKENIZER, '답변') + [EOS]
    expected_prefix = [BOS] + encode_text(TOKENIZER, render_prompt('질문'))
    assert encoded[:boundary].tolist() == expected_prefix
    assert TOKENIZER.decode(actual_targets[:-1]) == '답변'
    assert not response_mask(0, boundary-1, boundary).any()
    assert response_mask(boundary-1, 1, boundary)[0]  # 첫 답변 토큰이 loss에 포함됨
    print('PASS: speaker mapping, turn merge, monologue exclusion, response-only mask + EOS')

check_response_format()
eval_ids = sorted(val_ids, key=lambda i: pairs[i]['sha256'])[:CFG['eval_max_pairs']]


@torch.no_grad()
def evaluate(m):
    was_training = m.training
    m.eval()
    loss_sum, correct, total = 0., 0, 0
    try:
        for start in range(0, len(eval_ids), CFG['batch_size']):
            state = None
            for x, labels, active, counts in batched_chunks(eval_ids[start:start+CFG['batch_size']], CFG['eval_response_tokens']):
                logits, state = m(x, state, active_mask=active)
                supervised = int(counts.sum())
                if supervised:
                    loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                                ignore_index=IGNORE_INDEX, reduction='sum').item()
                    correct += int(((logits.argmax(-1) == labels) & (labels != IGNORE_INDEX)).sum())
                    total += supervised
    finally:
        m.train(was_training)
    mean = loss_sum / total
    if not math.isfinite(mean):
        raise RuntimeError('Nonfinite validation loss')
    return dict(response_loss=mean, response_perplexity=math.exp(mean) if mean < 700 else None,
                response_accuracy=correct/total, response_tokens=total, pairs=len(eval_ids),
                full_validation=total == data_stats['validation_response_tokens'])

@torch.no_grad()
def probe_neural_activity(m, question='안녕하세요?', max_tokens=32):
    # 체크포인트 재계산과 분리된 순수 추론 경로에서 실제 이산 사건을 집계한다.
    was_training = m.training
    m.eval()
    try:
        ids = [m.bos_id] + encode_text(m.tokenizer, render_prompt(question))
        ids = ids[:max_tokens]
        state = m.initial_state(1)
        weights = m.effective_weights()
        alpha, threshold, adaptation = m.dynamics()
        count = state.new_zeros(m.n)
        for token in ids:
            state, events = m.advance(torch.tensor([token], device=state.device), state,
                                      weights, alpha, threshold, adaptation, collect_spikes=True)
            count += events.sum(0)
        spiking = m.spiking_mask
        selected = count[spiking]
        ticks = len(ids) * m.k
        result = dict(probe_question=question, probe_tokens=len(ids), simulation_ticks=ticks,
                      spiking_neurons=int(spiking.sum()),
                      spike_events=int(selected.sum()),
                      spikes_per_neuron_tick=float(selected.sum() / (ticks * selected.numel()))
                          if selected.numel() else None,
                      silent_spiking_fraction=float((selected == 0).float().mean())
                          if selected.numel() else None,
                      mean_synaptic_signal=float(state[1].mean()),
                      finite_state=bool(torch.isfinite(state).all()))
        if not result['finite_state']:
            raise RuntimeError('신경 활동 진단에서 유한하지 않은 상태를 발견했다.')
        return result
    finally:
        m.train(was_training)


def print_activity_probe(result):
    print('신경 활동 진단:', json.dumps(result, ensure_ascii=False))
    if result['spiking_neurons'] and result['silent_spiking_fraction'] == 1.0:
        print('주의: 이 짧은 입력에서 모든 스파이크 유형이 침묵했다. 전체 학습의 침묵으로 단정하지 말고 여러 입력을 확인하세요.')

def to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu(item) for item in value)
    return value


# 고정된 전체 그래프를 한 번 내보내 체크포인트마다 중복 저장하지 않는다.
graph_export = RUN_DIR/'graph'
graph_export.mkdir(exist_ok=True)
for filename in ['src.npy','count.npy','crow.npy','body_ids.npy','complete.json']:
    if not (graph_export/filename).exists():
        shutil.copyfile(full_cache/filename,graph_export/filename)
np.save(graph_export/'edge_sign.npy',edge_sign)
np.save(graph_export/'inputs.npy',input_nodes)
np.save(graph_export/'outputs.npy',output_nodes)
np.save(graph_export/'type_ids.npy',type_ids)
np.save(graph_export/'dynamics_ids.npy', dynamics_ids)
for filename in ('dynamics_evidence.json', 'neuron_activity_annotations.csv',
                 'activity_types.csv', 'activity_coverage.json'):
    shutil.copyfile(RUN_DIR/filename, graph_export/filename)
dynamics_ids_sha256 = hashlib.sha256(dynamics_ids.tobytes()).hexdigest()
body_ids_sha256 = hashlib.sha256(np.asarray(body_ids, dtype=np.int64).tobytes()).hexdigest()
(graph_export/'type_names.json').write_text(json.dumps(type_names, ensure_ascii=False), encoding='utf-8')

def save_checkpoint(name,m,optimizer,updates,validation):
    # 모델 가중치만 저장하며 optimizer 상태와 데이터 위치는 저장하지 않는다.
    payload = dict(architecture='malecns_mixed_sentencepiece_v2', config=CFG,model_state=to_cpu(m.state_dict()),graph_directory='graph',
                   extract_spec=extract_spec,sources=source_manifest,data_spec=data_spec,
                   data_config=DATA_CFG,updates=updates,validation=validation,
                   dynamics_sha256=dynamics_sha256, dynamics_ids_sha256=dynamics_ids_sha256,
                   body_ids_sha256=body_ids_sha256, activity_coverage=activity_coverage,
                   tokenizer_sha256=tokenizer_sha256, tokenizer_spec=tokenizer_spec)
    local = CACHE/name
    torch.save(payload,local)
    temporary = RUN_DIR/(name+'.partial')
    shutil.copyfile(local,temporary)
    temporary.replace(RUN_DIR/name)
    del payload

def save_history(rows):
    (RUN_DIR / 'history.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
model = make_model()  # 재실행은 새 학습이다. 자동 탐색 모델은 사용하지 않는다.
optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'], foreach=False)
initial = dict(evaluate(model), evaluated_at_update=0)
best_loss, last_validation = initial['response_loss'], initial
initial_activity = probe_neural_activity(model)
print_activity_probe(initial_activity)
activity_history = [dict(update=0, **initial_activity)]
history = [dict(update=0, batch_size=CFG['batch_size'], validation=initial)]
save_checkpoint('best.pt', model, optimizer, 0, initial)
save_history(history)
updates, seen_response_tokens, processed_tokens, completed_epochs = 0, 0, 0, 0
pairs_started, pairs_completed = 0, 0
training_seconds, window_loss, window_tokens = 0.0, 0.0, 0
planned_updates = sum(batch_updates(group) for epoch in range(CFG['epochs']) for group in epoch_batches(epoch))
if CFG['train_updates'] is not None:
    planned_updates = min(planned_updates, CFG['train_updates'])
torch.cuda.reset_peak_memory_stats()
wall_start = time.perf_counter()
with tqdm(total=planned_updates, desc=f"Train batch={CFG['batch_size']}") as progress:
    for epoch in range(CFG['epochs']):
        finished_in_epoch = 0
        for indices in epoch_batches(epoch):
            if updates >= planned_updates:
                break
            state = None
            response_seen = np.zeros(len(indices), dtype=np.int64)
            pairs_started += len(indices)
            for x, labels, active, counts in batched_chunks(indices):
                if updates >= planned_updates:
                    break
                model.train()
                supervised = int(counts.sum())
                torch.cuda.synchronize()
                tick = time.perf_counter()
                if supervised == 0:
                    with torch.no_grad():
                        _, state = model(x, state, active_mask=active)
                    state = state.detach()
                    torch.cuda.synchronize()
                    training_seconds += time.perf_counter() - tick
                    processed_tokens += x.numel()
                    continue
                optimizer.zero_grad(set_to_none=True)
                logits, next_state = model(x, state, active_mask=active)
                loss = F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1), ignore_index=IGNORE_INDEX)
                if not torch.isfinite(loss):
                    raise RuntimeError(f'Nonfinite loss at batch update {updates+1}')
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
                state = next_state.detach()
                loss_value, grad_value = loss.item(), float(grad_norm)
                # 평가 또는 다음 배치 전에 현재 청크의 계산 그래프와 기울기를 해제한다.
                del logits, next_state, loss, grad_norm
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.synchronize()
                training_seconds += time.perf_counter() - tick
                updates += 1
                processed_tokens += x.numel()
                seen_response_tokens += supervised
                response_seen += counts
                window_loss += loss_value * supervised
                window_tokens += supervised
                progress.update(1)
                if updates == 1 or updates % CFG['log_every'] == 0:
                    row = dict(update=updates, epoch=epoch+1, actual_batch=len(indices),
                               response_tokens=seen_response_tokens, train_response_loss=window_loss/window_tokens,
                               grad_norm=grad_value, response_tokens_per_second=seen_response_tokens/training_seconds,
                               peak_allocated_GiB=torch.cuda.max_memory_allocated()/2**30,
                               peak_reserved_GiB=torch.cuda.max_memory_reserved()/2**30)
                    history.append(row)
                    tqdm.write(json.dumps(row))
                    window_loss, window_tokens = 0.0, 0
                if updates % CFG['eval_every'] == 0:
                    last_validation = dict(evaluate(model), evaluated_at_update=updates)
                    activity = probe_neural_activity(model)
                    print_activity_probe(activity)
                    activity_history.append(dict(update=updates, **activity))
                    (RUN_DIR/'activity_history.json').write_text(
                        json.dumps(activity_history, ensure_ascii=False, indent=2), encoding='utf-8')
                    history.append(dict(update=updates, validation=last_validation))
                    tqdm.write(json.dumps(dict(update=updates, validation=last_validation)))
                    if last_validation['response_loss'] < best_loss:
                        best_loss = last_validation['response_loss']
                        save_checkpoint('best.pt', model, optimizer, updates, last_validation)
                if updates % CFG['save_every'] == 0:
                    save_checkpoint('latest.pt', model, optimizer, updates, last_validation)
                    save_history(history)
            completed = sum(int(seen == pairs[index]['length'] - pairs[index]['prompt_tokens'])
                            for index, seen in zip(indices, response_seen))
            pairs_completed += completed
            finished_in_epoch += completed
            del state
        if finished_in_epoch == len(train_ids):
            completed_epochs += 1
        if updates >= planned_updates:
            break
final = dict(evaluate(model), evaluated_at_update=updates)
history.append(dict(update=updates, validation=final, final=True))
if final['response_loss'] < best_loss:
    best_loss = final['response_loss']
    save_checkpoint('best.pt', model, optimizer, updates, final)
save_checkpoint('latest.pt', model, optimizer, updates, final)
save_history(history)
torch.cuda.synchronize()
report = dict(batch_size=CFG['batch_size'], initial_validation=initial, final_validation=final,
              best_response_loss=best_loss, updates=updates, train_response_tokens=seen_response_tokens,
              processed_model_tokens_including_padding=processed_tokens,
              pairs_started=pairs_started, pairs_completed=pairs_completed, completed_epochs=completed_epochs,
              stop_reason='epochs_completed' if completed_epochs == CFG['epochs'] else 'update_limit',
              training_seconds=training_seconds, response_tokens_per_second=seen_response_tokens/training_seconds,
              wall_seconds=time.perf_counter()-wall_start,
              peak_allocated_MiB=torch.cuda.max_memory_allocated()/2**20,
              peak_reserved_MiB=torch.cuda.max_memory_reserved()/2**20)
print(json.dumps(report, ensure_ascii=False, indent=2))
final_activity = probe_neural_activity(model)
print_activity_probe(final_activity)
activity_history.append(dict(update=updates, final=True, **final_activity))
(RUN_DIR/'activity_history.json').write_text(
    json.dumps(activity_history, ensure_ascii=False, indent=2), encoding='utf-8')

## 8. SentencePiece 생성과 체크포인트 복원

질문을 한 번 읽고 생성한 토큰과 네 신경 상태만 전달한다. 출력 길이는 `max_new_tokens`로 지정한다.
일반 subword와 fallback 바이트를 모두 생성할 수 있다. UNK/BOS/PAD는 출력에서 제외한다.
fallback 문자가 완성되기 전에는 이어지는 유효 UTF-8 바이트만 허용하며, 미완성 끝 문자는 생략한다.
최종 출력은 저장된 SentencePiece 모델로 디코딩한다.

복원 시 아키텍처·토크나이저 모델 해시/어휘/특수 토큰·신경 근거집·활동 코드·뉴런 순서를 확인한다.
다음 ZIP에는 `tokenizer.model`, `tokenizer.json`, 체크포인트, 그래프, 동일 구현의 추론 모듈이 포함된다.

In [ ]:
import codecs


def inference_inputs(m, question):
    question = ' '.join(question.split())
    if not question:
        raise ValueError('비어 있지 않은 질문이 필요하다.')
    return [m.bos_id] + encode_text(m.tokenizer, render_prompt(question))


@torch.no_grad()
def prefill_question(m, question):
    prefix = inference_inputs(m, question)
    state = None
    for start in range(0, len(prefix), m.tbptt):
        ids = torch.tensor([prefix[start:start + m.tbptt]], device=m.embedding.weight.device)
        logits, state = m(ids, state)
    return logits, state


def valid_utf8_bytes(pending):
    if not pending:
        return list(range(0x80)) + list(range(0xC2, 0xF5))
    if len(pending) == 1:
        bounds = {0xE0: (0xA0, 0xC0), 0xED: (0x80, 0xA0),
                  0xF0: (0x90, 0xC0), 0xF4: (0x80, 0x90)}
        if pending[0] in bounds:
            return list(range(*bounds[pending[0]]))
    return list(range(0x80, 0xC0))


def generation_tables(tokenizer):
    regular, byte_ids, byte_values = [], {}, {}
    for index in range(tokenizer.get_piece_size()):
        if tokenizer.is_byte(index):
            value = int(tokenizer.id_to_piece(index)[3:5], 16)
            byte_ids[value] = index
            byte_values[index] = value
        elif not (tokenizer.is_control(index) or tokenizer.is_unknown(index) or tokenizer.is_unused(index)):
            regular.append(index)
    return regular, byte_ids, byte_values


def allowed_piece_ids(m, decoder, tables):
    regular, byte_ids, _ = tables
    pending = decoder.getstate()[0]
    # fallback 바이트로 시작한 문자가 완성되기 전에는 다른 subword나 EOS를 허용하지 않는다.
    allowed = [byte_ids[value] for value in valid_utf8_bytes(pending)]
    if not pending:
        allowed.extend(regular)
        allowed.append(m.eos_id)
    return allowed


@torch.no_grad()
def reply(m, question, max_new_tokens=128, temperature=0.8):
    if type(max_new_tokens) is not int or max_new_tokens < 1 or not math.isfinite(temperature) or temperature < 0:
        raise ValueError('max_new_tokens는 양의 정수, temperature는 유한한 0 이상의 값이어야 한다.')
    was_training = m.training
    m.eval()
    decoder = codecs.getincrementaldecoder('utf-8')('strict')
    generated = []
    tables = generation_tables(m.tokenizer)
    try:
        logits, state = prefill_question(m, question)
        for _ in range(max_new_tokens):
            allowed = torch.tensor(allowed_piece_ids(m, decoder, tables), device=logits.device)
            scores = logits[0, -1].index_select(0, allowed)
            if not torch.isfinite(scores).all():
                raise RuntimeError('생성 점수에 유한하지 않은 값이 있다.')
            selected = scores.argmax() if temperature == 0 else torch.multinomial(
                (scores / temperature).softmax(-1), 1)[0]
            token = int(allowed[selected])
            if token == m.eos_id:
                break
            generated.append(token)
            if token in tables[2]:
                decoder.decode(bytes([tables[2][token]]), final=False)
            logits, state = m(torch.tensor([[token]], device=logits.device), state)
        # 출력 한도에서 끝나지 않은 fallback 문자는 해당 바이트 토큰만 생략한다.
        pending = decoder.getstate()[0]
        if pending:
            generated = generated[:-len(pending)]
        return m.tokenizer.decode(generated)
    finally:
        m.train(was_training)

def load_model(checkpoint_path, device=None):
    checkpoint_path = Path(checkpoint_path)
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if saved.get('architecture') != 'malecns_mixed_sentencepiece_v2':
        raise ValueError('SentencePiece용 flylm-2 체크포인트가 필요하다. 바이트 모델은 직접 호환되지 않는다.')
    if saved.get('graph_directory') != 'graph':
        raise ValueError('이 버전은 체크포인트 옆의 graph 폴더만 사용한다.')
    tokenizer_bytes = (checkpoint_path.parent/'tokenizer.model').read_bytes()
    if hashlib.sha256(tokenizer_bytes).hexdigest() != saved['tokenizer_sha256']:
        raise ValueError('체크포인트와 토크나이저 모델의 해시가 다르다.')
    tokenizer = spm.SentencePieceProcessor(model_proto=tokenizer_bytes)
    validate_tokenizer(tokenizer, saved['config'])
    folder = checkpoint_path.parent / 'graph' 
    evidence_bytes = (folder/'dynamics_evidence.json').read_bytes()
    if hashlib.sha256(evidence_bytes).hexdigest() != saved['dynamics_sha256']:
        raise ValueError('체크포인트와 신경 활동 근거집의 해시가 다르다.')
    evidence = json.loads(evidence_bytes)
    if (evidence['mode'] != saved['config']['neuron_mode']
            or evidence['unknown_activity'] != saved['config']['unknown_activity']):
        raise ValueError('체크포인트와 근거집의 동역학 모드가 다르다.')
    codes = np.load(folder/'dynamics_ids.npy', allow_pickle=False)
    body_ids = np.load(folder/'body_ids.npy', allow_pickle=False)
    if hashlib.sha256(codes.tobytes()).hexdigest() != saved['dynamics_ids_sha256']:
        raise ValueError('체크포인트와 신경 활동 배정이 다르다.')
    if hashlib.sha256(body_ids.tobytes()).hexdigest() != saved['body_ids_sha256']:
        raise ValueError('체크포인트와 뉴런 순서가 다르다.')
    info = json.loads((folder/'complete.json').read_text(encoding='utf-8'))
    count = info['nnz']
    if len(body_ids) != saved['config']['n_neurons']:
        raise ValueError('체크포인트와 그래프의 뉴런 수가 다르다.')
    target = torch.device(device if device is not None else ('cuda' if torch.cuda.is_available() else 'cpu'))
    restored = ConnectomeLM(saved['config'], np.load(folder/'crow.npy', mmap_mode='r'),
        np.load(folder/'src.npy', mmap_mode='r')[:count],
        np.load(folder/'count.npy', mmap_mode='r')[:count],
        np.load(folder/'edge_sign.npy', mmap_mode='r'),
        np.load(folder/'inputs.npy'), np.load(folder/'outputs.npy'),
        np.load(folder/'type_ids.npy'), codes, target)
    restored.tokenizer = tokenizer
    restored.load_state_dict(saved['model_state'], strict=True)
    return restored.eval()


def load_full_checkpoint(checkpoint_path):
    return load_model(checkpoint_path, device=device)

# 최고 검증 모델을 불러오기 전에 Adam과 현재 모델을 해제한다.
del optimizer, model
gc.collect()
torch.cuda.empty_cache()
model = load_full_checkpoint(RUN_DIR/'best.pt')
examples = [dict(input=q, output=reply(model, q)) for q in
            ['안녕하세요?', '좋아하는 음식은 뭐예요?', '기차는 어디에서 타나요?']]
print(json.dumps(examples, ensure_ascii=False, indent=2))
(RUN_DIR/'examples.json').write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR/'result.json').write_text(json.dumps(dict(config=CFG, versions=VERSIONS,
    sources=source_manifest, data_config=DATA_CFG, data_spec=data_spec, data_stats=data_stats,
    graph=graph_stats, dynamics_groups=len(type_names), metrics=report,
    activity_coverage=activity_coverage, final_activity=final_activity,
    dynamics_sha256=dynamics_sha256, tokenizer_sha256=tokenizer_sha256, tokenizer_spec=tokenizer_spec),
    ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', RUN_DIR)

# print(reply(model, '오늘 기분이 어때요?', temperature=0))

In [ ]:
INFERENCE_SOURCE = r'''
# FlyLM-2 SentencePiece 혼합 동역학 추론 모듈. 학습 노트북과 함께 내보낸다.
import codecs
import hashlib
import json
import math
from pathlib import Path
import numpy as np
import sentencepiece as spm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

def encode_text(tokenizer, text):
    # SentencePiece의 공백 표식과 실제 U+2581 문자를 구분해 원문을 보존한다.
    if '▁' not in text:
        return tokenizer.encode(text, out_type=int)
    parts = text.split('▁')
    ids = []
    literal = [tokenizer.piece_to_id(f'<0x{value:02X}>') for value in '▁'.encode('utf-8')]
    for i, part in enumerate(parts):
        if i:
            ids.extend(literal)
        ids.extend(tokenizer.encode(part, out_type=int))
    return ids


def render_prompt(question):
    return '화자1: ' + question + '\n화자2: '


def tokenizer_metadata(tokenizer):
    return dict(vocab_size=tokenizer.get_piece_size(), bos_id=tokenizer.bos_id(),
                eos_id=tokenizer.eos_id(), pad_id=tokenizer.pad_id(), unk_id=tokenizer.unk_id())


def validate_tokenizer(tokenizer, expected=None):
    meta = tokenizer_metadata(tokenizer)
    if (meta['bos_id'], meta['eos_id'], meta['pad_id'], meta['unk_id']) != (1, 2, 3, 0):
        raise ValueError('특수 토큰 설정이 다르다: BOS=1, EOS=2, PAD=3, UNK=0이어야 한다.')
    if not 260 <= meta['vocab_size'] <= 65536:
        raise ValueError('바이트 fallback과 uint16 캐시에 맞는 어휘 크기가 필요하다.')
    for value in range(256):
        piece = f'<0x{value:02X}>'
        index = tokenizer.piece_to_id(piece)
        if not tokenizer.is_byte(index) or tokenizer.id_to_piece(index) != piece:
            raise ValueError('모든 256개 바이트 fallback 토큰이 필요하다.')
    if expected is not None and any(expected[key] != value for key, value in meta.items()):
        raise ValueError('토크나이저의 어휘/특수 토큰 정보와 체크포인트 설정이 다르다.')
    return meta

class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, voltage_gap, slope):
        ctx.save_for_backward(voltage_gap)
        ctx.slope = slope
        return (voltage_gap >= 0).to(voltage_gap.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (gap,) = ctx.saved_tensors
        # 순전파는 이산 사건이며, 역전파에만 빠른 시그모이드의 대리 미분을 쓴다.
        return grad_output / (1 + ctx.slope * gap.abs()).square(), None


def mixed_neuron_tick(v, signal, slow, refractory, current, alpha, threshold,
                      adaptation, spiking_mask, reset_voltage, refractory_ticks, surrogate_slope):
    candidate = v + alpha[0] * (-v + current - adaptation * slow)
    blocked = spiking_mask & (refractory > 0)
    candidate = torch.where(blocked, torch.full_like(candidate, reset_voltage), candidate)
    spikes = SurrogateSpike.apply(candidate - threshold, surrogate_slope)
    spikes = spikes * (spiking_mask & ~blocked).to(candidate.dtype)
    graded = torch.relu(torch.tanh(candidate - threshold))
    release = torch.where(spiking_mask, spikes, graded)
    # 리셋·불응기 사건의 이산 결정에는 별도 대리 미분을 적용하지 않는다.
    fired = spikes.detach() > 0
    v = torch.where(fired, torch.full_like(candidate, reset_voltage), candidate)
    refractory = torch.where(fired, torch.full_like(refractory, float(refractory_ticks)),
                             torch.clamp(refractory - 1, min=0))
    refractory = torch.where(spiking_mask, refractory, torch.zeros_like(refractory)).detach()
    signal = signal + alpha[1] * (release - signal)
    slow = slow + alpha[2] * (release - slow)
    return v, signal, slow, refractory, spikes


def validate_dynamics_config(cfg):
    if cfg['neuron_mode'] not in ('mixed', 'graded_control'):
        raise ValueError('neuron_mode은 mixed 또는 graded_control이어야 한다.')
    if cfg['unknown_activity'] not in ('graded', 'spiking'):
        raise ValueError('미확인 유형의 기본 동역학은 graded 또는 spiking이어야 한다.')
    for key in ('spike_threshold_center', 'spike_threshold_radius', 'spike_reset', 'surrogate_slope'):
        if not math.isfinite(cfg[key]):
            raise ValueError(f'유한한 동역학 설정이 필요하다: {key}')
    if (cfg['spike_threshold_radius'] < 0 or cfg['surrogate_slope'] <= 0
            or cfg['spike_threshold_center'] - cfg['spike_threshold_radius'] <= max(0, cfg['spike_reset'])):
        raise ValueError('스파이크 역치는 항상 0과 리셋 전압보다 높아야 한다.')
    if (type(cfg['refractory_ticks']) is not int or cfg['refractory_ticks'] < 0
            or type(cfg['brain_steps']) is not int or cfg['brain_steps'] < 1):
        raise ValueError('불응기는 0 이상의 정수, 토큰당 tick 수는 양의 정수여야 한다.')


class ConnectomeLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, dynamics_ids, device):
        super().__init__()
        validate_dynamics_config(cfg)
        self.tbptt = cfg['tbptt']
        self.vocab_size = cfg['vocab_size']
        self.bos_id, self.eos_id = cfg['bos_id'], cfg['eos_id']
        self.pad_id, self.unk_id = cfg['pad_id'], cfg['unk_id']
        self.tokenizer = None
        self.reset_voltage = cfg['spike_reset']
        self.refractory_ticks = cfg['refractory_ticks']
        self.surrogate_slope = cfg['surrogate_slope']
        self.spike_threshold_center = cfg['spike_threshold_center']
        self.spike_threshold_radius = cfg['spike_threshold_radius']
        codes = np.asarray(dynamics_ids)
        if (codes.shape != (cfg['n_neurons'],) or codes.dtype != np.uint8
                or not np.isin(codes, [0, 1, 2]).all()):
            raise ValueError('뉴런 순서에 맞는 uint8 활동 코드 0/1/2가 필요하다.')
        mask = (codes == 2) & (cfg['neuron_mode'] == 'mixed')
        self.register_buffer('spiking_mask', torch.tensor(mask, dtype=torch.bool, device=device), persistent=False)
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        d = cfg['embedding_dim']
        self.embedding = nn.Embedding(self.vocab_size, d, device=device)
        self.input_adapter = nn.Linear(d, len(inputs), bias=False, device=device)
        self.motor_norm = nn.LayerNorm(len(outputs), eps=1e-6, elementwise_affine=False, device=device)
        self.output_adapter = nn.Linear(len(outputs), d, bias=False, device=device)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        offset = self.threshold.tanh()[self.type_ids]
        threshold = torch.where(self.spiking_mask,
            self.spike_threshold_center + self.spike_threshold_radius * offset, 0.5 * offset)
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return self.embedding.weight.new_zeros(4, batch, self.n)

    def advance(self, token, state, weights, alpha, threshold, adaptation, collect_spikes=False):
        v, signal, slow, refractory = state.unbind(0)
        value = self.input_adapter(self.embedding(token))
        drive = torch.zeros_like(v).index_add(1, self.inputs, value)
        counts = torch.zeros_like(v) if collect_spikes else None
        for _ in range(self.k):
            message = SparseEdges.apply(signal, weights, self.rowptr, self.src, self.dst, self.block)
            v, signal, slow, refractory, spikes = mixed_neuron_tick(
                v, signal, slow, refractory, message + drive, alpha, threshold, adaptation,
                self.spiking_mask, self.reset_voltage, self.refractory_ticks, self.surrogate_slope)
            if collect_spikes:
                counts = counts + spikes.detach()
        result = torch.stack((v, signal, slow, refractory))
        return (result, counts) if collect_spikes else result

    def forward(self, tokens, state=None, active_mask=None):
        if tokens.ndim != 2 or tokens.shape[1] == 0:
            raise ValueError('Nonempty tokens [B,T] required')
        state = self.initial_state(tokens.shape[0]) if state is None else state
        if state.shape != (4, tokens.shape[0], self.n):
            raise ValueError('상태는 [4,B,N]: 막전위, 시냅스 신호, 적응, 남은 불응기여야 한다.')
        if active_mask is not None and (active_mask.shape != tokens.shape or active_mask.dtype != torch.bool):
            raise ValueError('active_mask must be bool [B,T]')
        weights = self.effective_weights()
        alpha, threshold, adaptation = self.dynamics()
        logits = []
        for t in range(tokens.shape[1]):
            previous = state
            args = (tokens[:, t], state, weights, alpha, threshold, adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                state = checkpoint(self.advance, *args, use_reentrant=False, preserve_rng_state=False)
            else:
                state = self.advance(*args)
            if active_mask is not None:
                state = torch.where(active_mask[:, t][None, :, None], state, previous)
            motor = self.motor_norm(state[1].index_select(1, self.outputs))
            logits.append(F.linear(self.output_adapter(motor), self.embedding.weight))
        return torch.stack(logits, dim=1), state

import codecs


def inference_inputs(m, question):
    question = ' '.join(question.split())
    if not question:
        raise ValueError('비어 있지 않은 질문이 필요하다.')
    return [m.bos_id] + encode_text(m.tokenizer, render_prompt(question))


@torch.no_grad()
def prefill_question(m, question):
    prefix = inference_inputs(m, question)
    state = None
    for start in range(0, len(prefix), m.tbptt):
        ids = torch.tensor([prefix[start:start + m.tbptt]], device=m.embedding.weight.device)
        logits, state = m(ids, state)
    return logits, state


def valid_utf8_bytes(pending):
    if not pending:
        return list(range(0x80)) + list(range(0xC2, 0xF5))
    if len(pending) == 1:
        bounds = {0xE0: (0xA0, 0xC0), 0xED: (0x80, 0xA0),
                  0xF0: (0x90, 0xC0), 0xF4: (0x80, 0x90)}
        if pending[0] in bounds:
            return list(range(*bounds[pending[0]]))
    return list(range(0x80, 0xC0))


def generation_tables(tokenizer):
    regular, byte_ids, byte_values = [], {}, {}
    for index in range(tokenizer.get_piece_size()):
        if tokenizer.is_byte(index):
            value = int(tokenizer.id_to_piece(index)[3:5], 16)
            byte_ids[value] = index
            byte_values[index] = value
        elif not (tokenizer.is_control(index) or tokenizer.is_unknown(index) or tokenizer.is_unused(index)):
            regular.append(index)
    return regular, byte_ids, byte_values


def allowed_piece_ids(m, decoder, tables):
    regular, byte_ids, _ = tables
    pending = decoder.getstate()[0]
    # fallback 바이트로 시작한 문자가 완성되기 전에는 다른 subword나 EOS를 허용하지 않는다.
    allowed = [byte_ids[value] for value in valid_utf8_bytes(pending)]
    if not pending:
        allowed.extend(regular)
        allowed.append(m.eos_id)
    return allowed


@torch.no_grad()
def reply(m, question, max_new_tokens=128, temperature=0.8):
    if type(max_new_tokens) is not int or max_new_tokens < 1 or not math.isfinite(temperature) or temperature < 0:
        raise ValueError('max_new_tokens는 양의 정수, temperature는 유한한 0 이상의 값이어야 한다.')
    was_training = m.training
    m.eval()
    decoder = codecs.getincrementaldecoder('utf-8')('strict')
    generated = []
    tables = generation_tables(m.tokenizer)
    try:
        logits, state = prefill_question(m, question)
        for _ in range(max_new_tokens):
            allowed = torch.tensor(allowed_piece_ids(m, decoder, tables), device=logits.device)
            scores = logits[0, -1].index_select(0, allowed)
            if not torch.isfinite(scores).all():
                raise RuntimeError('생성 점수에 유한하지 않은 값이 있다.')
            selected = scores.argmax() if temperature == 0 else torch.multinomial(
                (scores / temperature).softmax(-1), 1)[0]
            token = int(allowed[selected])
            if token == m.eos_id:
                break
            generated.append(token)
            if token in tables[2]:
                decoder.decode(bytes([tables[2][token]]), final=False)
            logits, state = m(torch.tensor([[token]], device=logits.device), state)
        # 출력 한도에서 끝나지 않은 fallback 문자는 해당 바이트 토큰만 생략한다.
        pending = decoder.getstate()[0]
        if pending:
            generated = generated[:-len(pending)]
        return m.tokenizer.decode(generated)
    finally:
        m.train(was_training)

def load_model(checkpoint_path, device=None):
    checkpoint_path = Path(checkpoint_path)
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if saved.get('architecture') != 'malecns_mixed_sentencepiece_v2':
        raise ValueError('SentencePiece용 flylm-2 체크포인트가 필요하다. 바이트 모델은 직접 호환되지 않는다.')
    if saved.get('graph_directory') != 'graph':
        raise ValueError('이 버전은 체크포인트 옆의 graph 폴더만 사용한다.')
    tokenizer_bytes = (checkpoint_path.parent/'tokenizer.model').read_bytes()
    if hashlib.sha256(tokenizer_bytes).hexdigest() != saved['tokenizer_sha256']:
        raise ValueError('체크포인트와 토크나이저 모델의 해시가 다르다.')
    tokenizer = spm.SentencePieceProcessor(model_proto=tokenizer_bytes)
    validate_tokenizer(tokenizer, saved['config'])
    folder = checkpoint_path.parent / 'graph' 
    evidence_bytes = (folder/'dynamics_evidence.json').read_bytes()
    if hashlib.sha256(evidence_bytes).hexdigest() != saved['dynamics_sha256']:
        raise ValueError('체크포인트와 신경 활동 근거집의 해시가 다르다.')
    evidence = json.loads(evidence_bytes)
    if (evidence['mode'] != saved['config']['neuron_mode']
            or evidence['unknown_activity'] != saved['config']['unknown_activity']):
        raise ValueError('체크포인트와 근거집의 동역학 모드가 다르다.')
    codes = np.load(folder/'dynamics_ids.npy', allow_pickle=False)
    body_ids = np.load(folder/'body_ids.npy', allow_pickle=False)
    if hashlib.sha256(codes.tobytes()).hexdigest() != saved['dynamics_ids_sha256']:
        raise ValueError('체크포인트와 신경 활동 배정이 다르다.')
    if hashlib.sha256(body_ids.tobytes()).hexdigest() != saved['body_ids_sha256']:
        raise ValueError('체크포인트와 뉴런 순서가 다르다.')
    info = json.loads((folder/'complete.json').read_text(encoding='utf-8'))
    count = info['nnz']
    if len(body_ids) != saved['config']['n_neurons']:
        raise ValueError('체크포인트와 그래프의 뉴런 수가 다르다.')
    target = torch.device(device if device is not None else ('cuda' if torch.cuda.is_available() else 'cpu'))
    restored = ConnectomeLM(saved['config'], np.load(folder/'crow.npy', mmap_mode='r'),
        np.load(folder/'src.npy', mmap_mode='r')[:count],
        np.load(folder/'count.npy', mmap_mode='r')[:count],
        np.load(folder/'edge_sign.npy', mmap_mode='r'),
        np.load(folder/'inputs.npy'), np.load(folder/'outputs.npy'),
        np.load(folder/'type_ids.npy'), codes, target)
    restored.tokenizer = tokenizer
    restored.load_state_dict(saved['model_state'], strict=True)
    return restored.eval()

'''

from pathlib import Path
import shutil

# 검증 성능이 가장 좋았던 모델.
# 최종 학습본을 저장하려면 'latest.pt'로 변경하세요.
checkpoint_name = 'best.pt'

run_dir = Path(RUN_DIR)
checkpoint_path = run_dir / checkpoint_name
graph_dir = run_dir / 'graph'
export_dir = run_dir / 'flylm2_inference'

if not checkpoint_path.is_file():
    raise FileNotFoundError(f'모델 파일이 없습니다: {checkpoint_path}')
if not graph_dir.is_dir():
    raise FileNotFoundError(f'뇌 연결망 폴더가 없습니다: {graph_dir}')

export_dir.mkdir(parents=True, exist_ok=True)

shutil.copyfile(checkpoint_path, export_dir / 'model.pt')
shutil.copyfile(run_dir/'tokenizer.model', export_dir/'tokenizer.model')
shutil.copyfile(run_dir/'tokenizer.json', export_dir/'tokenizer.json')
shutil.copytree(
    graph_dir,
    export_dir / 'graph',
    dirs_exist_ok=True,
)

(export_dir / 'flylm_inference.py').write_text(INFERENCE_SOURCE, encoding='utf-8')
(export_dir / 'README.txt').write_text(
    'FlyLM-2 SentencePiece 혼합 동역학 추론 번들\n'
    'model.pt, tokenizer.model, graph/, flylm_inference.py를 같은 폴더에 유지하세요.\n'
    '이 번들의 모듈로 load_model(model.pt)를 호출한 뒤 reply(model, 질문)를 사용하세요.\n'
    '필요 패키지: numpy, torch, sentencepiece==0.2.2. 바이트 모델용 추론 모듈은 사용할 수 없습니다.\n'
    '출력 길이는 max_new_tokens로 지정합니다. 토크나이저 파일은 재학습하거나 교체하지 마세요.\n'
    '신경 상태는 [4,B,N]: 막전위, 시냅스 신호, 적응, 불응기입니다.\n'
    'graph/dynamics_evidence.json에 문헌 근거와 전이 가정을 저장했습니다.\n'
    '미확인 뉴런의 동역학은 저장된 배정을 따르며, 문헌상 확인된 신호 방식이 아닙니다.\n', encoding='utf-8')

zip_path = shutil.make_archive(
    str(export_dir),
    'zip',
    root_dir=run_dir,
    base_dir=export_dir.name,
)

print('저장 완료:', zip_path)
print('Kaggle Files에서 이 ZIP을 다운로드하세요.')

## 실행 결과 확인

- `batch_tuning.json`: 전체 그래프의 검사 배치, peak VRAM, 검사 시간.
- `neuron_activity_annotations.csv`: 모든 뉴런의 원본 유형, 근거 유무, 출처, 대응 가정, 실제 사용 방식.
- `activity_types.csv`, `activity_coverage.json`: 유형별 집계와 문헌 근거/미확인 비율.
- `dynamics_evidence.json`: 버전이 붙은 근거집과 상태 정의. 체크포인트에 해시를 함께 저장한다.
- `activity_history.json`: 고정된 짧은 입력의 실제 스파이크 수·침묵 비율·시냅스 신호. Hz나 생체 기록이 아니다.
- `history.json`, `result.json`: 학습 및 검증 지표. 기존 4쌍 검증은 작은 실행 점검이며 충분한 품질 평가가 아니다.
- `flylm2_inference.zip`: `model.pt`, `graph/`, `tokenizer.model`, `tokenizer.json`, SentencePiece용 `flylm_inference.py`, 한국어 사용 안내.

기본 근거집은 H2·DM1_lPN의 스파이크와 HSN/HSE/HSS·R1-R6의 연속 신호를 지지한다.
H1 등 문헌 확인이 완료되지 않은 유형에는 이름이나 영역만으로 방식을 확장하지 않는다.
미확인 뉴런의 연속 기본값은 해당 뉴런이 생물학적으로 비스파이킹이라는 뜻이 아니다.
근거가 적으므로 스파이크 뉴런 비율이 매우 작을 수 있다. 이것을 전체 뇌 생리학의 분포로 해석하지 않는다.

토크나이저 설정은 `TOK_CFG`, 실제 모델과 학습 데이터 범위는 `tokenizer.json`에서 확인한다.
어휘를 바꾸거나 다시 학습하면 기존 모델 가중치와 직접 호환되지 않는다. 전체 학습을 새로 시작한다.